<a href="https://colab.research.google.com/github/SebaSRomanH/CAPSTONE-UAI-CV-VIOLENCE/blob/main/Capstone_Vigia_X3D_EC2_T4_Colab_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VigIA · X3D-Medium (3 versiones) vs. baseline TSN para detección de violencia
**Colab (interfaz) → túnel SSH → EC2 `g4dn.xlarge` (Tesla T4 16 GB) → Amazon S3**

| Sección | Contenido |
|---|---|
| 1 | Activar la EC2 y conectar Colab (paso a paso) |
| 2 | Librerías |
| 3 | Funciones (video, datos, modelos, entrenamiento robusto, métricas, gráficos) |
| 4 | Parámetros del notebook (única celda editable) |
| 5 | Carga de datos, misma partición del TSN y extracción del banco de frames (detecta videos corruptos) |
| 6 | Arquitectura X3D-Medium y propuesta de 3 versiones |
| 7 | Entrenamiento V1 |
| 8 | Entrenamiento V2 |
| 9 | Entrenamiento V3 |
| 10 | Métricas de desempeño por época |
| 11 | Comparación de las 3 versiones y selección del mejor modelo |
| 12 | Aplicabilidad: 2 ejemplos de cada tipo (TP, TN, FP, FN) |
| 13 | Carga del modelo TSN |
| 14 | Evaluación TSN vs. mejor X3D-Medium en test |
| 15 | Comparación en los videos de la municipalidad |

### Alineación metodológica TSN vs. X3D-Medium

| Elemento | TSN V4 (`modelamiento_version4`) | X3D-Medium (este notebook) |
|---|---|---|
| Manifest | `df_manifest.csv` | el mismo archivo |
| Etiqueta | `anomalía` (columna usada por el script del TSN) | la misma |
| Partición | `train_test_split` estratificado, `random_state=42`: 70 / 15 / 15 | **reproducida idéntica**: mismos videos en train, val y test |
| Frames | $N=16$, segmentos homogéneos; aleatorio en train, central en val/test | igual |
| Videos con $F\le16$ frames | todos los frames y se repite el último | igual |
| Dimensiones | lado corto 256 (bilineal) + recorte central 224×224 | igual |
| Normalización | ImageNet | igual |
| Lectura de frames | decodifica el video en cada época | banco de frames extraído una sola vez (JPEG calidad 95) |
| Umbral | $\tau=0.5$ | igual |
| Selección | mejor AUC en validación | igual |

Diferencias que se declaran en el informe:
- El TSN entrenó con los videos ilegibles reemplazados por tensores en cero; aquí esos videos se detectan en la sección 5 y se excluyen.
- En entrenamiento, el frame aleatorio de cada segmento se sortea entre `CANDIDATOS_POR_SEGMENTO` frames guardados (8 por defecto), no entre todos los del segmento. En validación, test y municipal se usa exactamente el frame central, igual que el TSN.

En test, ambos modelos se evalúan sobre los mismos videos legibles y con los mismos 16 frames.

## 1. Activar la EC2 y conectar Colab (paso a paso)

Colab es solo la pantalla: el código corre en la EC2 con la Tesla T4. La llave está en:

`C:\Users\Usuario\Desktop\Capstone\Código\Modelamiento_seba.pem`

### Parte A. Solo la primera vez

**A1. Permisos de la llave.** Abrir **PowerShell** (menú Inicio → escribir "PowerShell" → Enter) y pegar:
```powershell
icacls "C:\Users\Usuario\Desktop\Capstone\Código\Modelamiento_seba.pem" /inheritance:r
icacls "C:\Users\Usuario\Desktop\Capstone\Código\Modelamiento_seba.pem" /grant:r "$($env:USERNAME):(R)"
```
Debe responder "Se procesaron correctamente 1 archivos".

**A2. Autorizar tu computador en AWS.**
1. Consola de AWS → **EC2** → **Instancias** → clic en tu instancia.
2. Pestaña **Seguridad** → clic en el grupo de seguridad (`sg-...`).
3. **Editar reglas de entrada** → agregar **Tipo: SSH** y **Origen: Mi IP** → **Guardar reglas**.

Si trabajas desde otra red (otra casa, la universidad), repite A2.

**A3. Instalar lo necesario en la EC2.**
1. Enciende la instancia (paso B1) y conéctate:
   ```powershell
   ssh -i "C:\Users\Usuario\Desktop\Capstone\Código\Modelamiento_seba.pem" ubuntu@IP_PUBLICA
   ```
   Si pregunta `yes/no`, escribe `yes`.
2. Ya dentro del servidor, pega:
   ```bash
   /opt/pytorch/bin/python -m pip install -U notebook boto3 fvcore iopath scikit-learn opencv-python-headless
   /opt/pytorch/bin/python -m pip install --no-deps pytorchvideo==0.1.5
   sudo apt-get install -y tmux
   exit
   ```

### Parte B. Cada vez que trabajes

**B1. Encender la instancia.** EC2 → **Instancias** → selecciona la instancia → **Estado de la instancia** → **Iniciar instancia**. Espera **En ejecución** y copia la **Dirección IPv4 pública** (cambia en cada encendido).

**B2. Conectarte con túnel** (PowerShell; no cierres esta ventana mientras trabajes):
```powershell
ssh -i "C:\Users\Usuario\Desktop\Capstone\Código\Modelamiento_seba.pem" -L 8888:localhost:8888 ubuntu@IP_PUBLICA
```

**B3. Iniciar Jupyter** (en esa misma ventana):
```bash
tmux new -A -s jupyter
/opt/pytorch/bin/python -m jupyter notebook --ServerApp.allow_origin='https://colab.research.google.com' --port=8888 --ServerApp.port_retries=0 --no-browser
```
Copia la dirección `http://localhost:8888/tree?token=...`. Si Jupyter ya estaba corriendo, `tmux new -A` entra a la sesión existente; si no ves la dirección, sal con `Ctrl+B` y luego `D`, y escribe `/opt/pytorch/bin/python -m jupyter notebook list`.

**B4. Conectar Colab.** Flecha junto a **Conectar** → **Conectar a un entorno de ejecución local** → pega la dirección → **Conectar**.

**B5. Verificar.** La celda de la sección 2 debe imprimir `Tesla T4` y el espacio libre en disco.

### Parte C. Al terminar

1. En PowerShell: `tmux attach -t jupyter` → `Ctrl+C` → `exit` dos veces.
2. EC2 → **Estado de la instancia** → **Detener instancia**. Nunca **Terminar**: borra el disco.

### Seguir el entrenamiento sin Colab
Si Colab se desconecta, el kernel sigue corriendo en la EC2 y los checkpoints se guardan en cada época. Para ver el avance desde otra ventana de PowerShell conectada por SSH:
```bash
tail -f ~/capstone_vigia/runs_v3/*/V1/train_log.txt
tail -f ~/capstone_vigia/banco_frames/*/extraccion_log.txt     # avance de la extracción de frames
```
(cambia `V1` por la versión que esté entrenando). Si el kernel se reinicia, vuelve a ejecutar el notebook: con `REANUDAR = True` cada versión continúa desde su última época guardada.

### Problemas frecuentes

| Mensaje | Solución |
|---|---|
| `Permission denied (publickey)` | Cambiar `ubuntu` por `ec2-user` |
| `Connection timed out` | Instancia apagada o IP no autorizada: repetir A2 |
| `UNPROTECTED PRIVATE KEY FILE` | Repetir A1 |
| `duplicate session: jupyter` | Usar `tmux new -A -s jupyter` |
| `No space left on device` | En la EC2: `df -h /` y `rm -rf ~/capstone_vigia/cache_videos` (son copias de S3; con el banco de frames ya no se necesitan) |

## 2. Librerías
Todas las importaciones en una sola celda. La variable `OPENCV_FFMPEG_LOGLEVEL` silencia avisos internos de FFmpeg (por ejemplo "Referenced QT chapter track not found"), que son inofensivos; los errores reales de lectura los detecta y registra el propio notebook.

In [ ]:
import os
os.environ['OPENCV_FFMPEG_LOGLEVEL'] = '-8'
import io, re, json, math, time, copy, random, pickle, hashlib, shutil, threading, warnings, traceback
from pathlib import Path
from itertools import combinations
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import boto3
from botocore.exceptions import BotoCoreError, ClientError, NoCredentialsError
import cv2
import matplotlib.pyplot as plt
from matplotlib import animation
from tqdm.auto import tqdm
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, roc_curve)
from IPython.display import display, Image

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

cv2.setNumThreads(1)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='torch')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEVICE == 'cuda'
libre = shutil.disk_usage(Path.home()).free / 1024**3
print('PyTorch', torch.__version__, '| dispositivo:', torch.cuda.get_device_name(0) if AMP else 'CPU (solo para depurar)',
      f'| disco libre: {libre:.1f} GB')

## 3. Funciones
Todas las funciones del notebook, agrupadas por tema. Solo se **definen** aquí; se ejecutan en las secciones siguientes, con los parámetros de la sección 4.

| Celda | Contenido |
|---|---|
| 3.1 | S3, caché local de videos y lectura de frames |
| 3.2 | Muestreo temporal, preprocesamiento, banco de frames, dataset y verificación de videos |
| 3.3 | Modelos: X3D-Medium y TSN V4 |
| 3.4 | Entrenamiento con controles anticaídas |
| 3.5 | Métricas y pruebas estadísticas |
| 3.6 | Gráficos, Grad-CAM y galería |

**Controles anticaídas del entrenamiento (3.4)**

| Riesgo | Control |
|---|---|
| Video corrupto o ilegible | La extracción del banco de frames (sección 5) abre cada video y registra el motivo de cada falla; durante el entrenamiento, un video que falla se **omite del lote** (no se reemplaza por ceros ni por otro video) y se registra en `fallos_lectura.csv` |
| Proceso del `DataLoader` colgado o caído | `timeout` en el `DataLoader`; la época se reintenta desde el estado guardado al inicio de esa época, hasta `MAX_REINTENTOS_EPOCA` veces |
| Memoria de GPU insuficiente (OOM) | el lote se descarta, se libera la memoria y se continúa; si se repite más de `MAX_OOM_POR_EPOCA` veces, se detiene con instrucción de bajar `BATCH_SIZE` |
| Pérdida `NaN` o infinita | el paso no se aplica y se cuenta; recorte de norma del gradiente (`GRAD_CLIP`) |
| Disco lleno | antes de cada época se verifica el espacio y se libera caché si hace falta |
| Corte de la instancia, Colab o kernel | `last.pt` al cierre de cada época (opcionalmente respaldado en S3); con `REANUDAR=True` continúa en la época siguiente |
| Seguimiento sin Colab | `train_log.txt` por versión y `extraccion_log.txt`, legibles con `tail -f` |
| Extracción interrumpida | cada video se guarda al terminarlo; al relanzar solo se procesan los que faltan (o se copian desde S3) |

In [ ]:
# ================== 3.1 S3, caché local y lectura de video ==================
_CACHE_LOCK = threading.Lock()
_S3_CLIENTS = {}
VIDEO_EXT = ('.avi', '.mpeg', '.mpg', '.mp4', '.mov', '.wmv', '.flv', '.mkv')


def s3():
    """Cliente S3 por proceso (seguro con DataLoader multiproceso)."""
    pid = os.getpid()
    if pid not in _S3_CLIENTS:
        _S3_CLIENTS[pid] = boto3.Session(region_name=AWS_REGION).client('s3')
    return _S3_CLIENTS[pid]


def s3_key_efectiva(key):
    """Con USAR_224 se leen las copias pre-convertidas a 224x224 (s3://.../videos_224/...)."""
    return PREFIJO_224 + os.path.splitext(key)[0] + '.mp4' if USAR_224 else key


def cache_path(key):
    k = s3_key_efectiva(key)
    return CACHE_DIR / f"{hashlib.sha1(k.encode()).hexdigest()[:16]}{Path(k).suffix.lower() or '.mp4'}"


def trim_cache(protected=None, min_free_gb=None, keep_recent_s=900):
    """LRU: respeta CACHE_MAX_GB y deja min_free_gb libres; no borra videos usados en los últimos 15 min."""
    min_free_gb = MIN_DISCO_LIBRE_GB if min_free_gb is None else min_free_gb
    with _CACHE_LOCK:
        files = []
        for p in CACHE_DIR.iterdir():
            try:
                if p.is_file() and '.part' not in p.name:
                    st = p.stat(); files.append((st.st_mtime, st.st_size, p))
            except FileNotFoundError:
                pass
        total, now, limit = sum(s for _, s, _ in files), time.time(), CACHE_MAX_GB * 1024**3
        for mtime, size, p in sorted(files):
            if total <= limit and shutil.disk_usage(CACHE_DIR).free > min_free_gb * 1024**3:
                break
            if p == protected or now - mtime < keep_recent_s:
                continue
            p.unlink(missing_ok=True); total -= size


def get_local_video(key, _retry=True):
    path = cache_path(key)
    if not path.is_file() or path.stat().st_size == 0:
        trim_cache(path)
        tmp = path.with_suffix(path.suffix + f'.part.{os.getpid()}.{threading.get_ident()}')
        try:
            s3().download_file(S3_BUCKET, s3_key_efectiva(key), str(tmp)); tmp.replace(path)
        finally:
            tmp.unlink(missing_ok=True)
    try:
        os.utime(path, None)
    except FileNotFoundError:
        if not _retry: raise
        return get_local_video(key, _retry=False)
    return path


def s3_sizes(prefixes):
    sizes = {}
    for prefix in prefixes:
        for page in s3().get_paginator('list_objects_v2').paginate(Bucket=S3_BUCKET, Prefix=prefix):
            sizes.update({o['Key']: o['Size'] for o in page.get('Contents', [])})
    return sizes


def video_meta(path):
    cap = cv2.VideoCapture(str(path))
    meta = (int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS),
            int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()
    n, fps, w, h = meta
    return n, (fps if fps and fps > 1 else 30.0), w, h


def resize_short_cv(frame, short):
    h, w = frame.shape[:2]; s = short / min(h, w)
    return cv2.resize(frame, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)


def read_frames(path, indices, short=None, fn=None):
    """Frames RGB uint8 en el orden de `indices`. `fn` se aplica a cada frame al leerlo (p. ej. recorte a 224,
    para no guardar en memoria frames de alta resolución); `short` reescala con OpenCV solo para visualizar."""
    n = video_meta(path)[0]
    wanted = sorted(set(int(i) for i in indices))
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened(): raise RuntimeError(f'No se pudo abrir {Path(path).name}')
    out = {}
    try:
        if n <= MAX_FRAMES_SECUENCIAL:
            target, i = set(wanted), 0
            while i <= wanted[-1] and cap.grab():
                if i in target:
                    ok, fr = cap.retrieve()
                    if ok: out[i] = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB) if fn is None else fn(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
                i += 1
        else:
            for i in wanted:
                cap.set(cv2.CAP_PROP_POS_FRAMES, i); ok, fr = cap.read()
                if ok: out[i] = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB) if fn is None else fn(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    finally:
        cap.release()
    missing = [i for i in wanted if i not in out]
    if not out or len(missing) > 0.25 * len(wanted):
        raise RuntimeError(f'{Path(path).name}: {len(missing)}/{len(wanted)} frames ilegibles')
    have = np.array(sorted(out))
    for i in missing:                       # conteo de frames inexacto en el contenedor
        out[i] = out[int(have[np.abs(have - i).argmin()])]
    frames = [out[int(i)] for i in indices]
    return [resize_short_cv(f, short) for f in frames] if short and fn is None else frames


def espacio_libre_gb():
    return shutil.disk_usage(CACHE_DIR).free / 1024**3

In [ ]:
# ================== 3.2 Muestreo, preprocesamiento, dataset y verificación ==================
def segment_indices(total, training, rng=None):
    """Igual que el TSN V4: N segmentos de largo L=floor(F/N); aleatorio en train, central en eval.
    Si F <= N se usan todos los frames y se repite el último."""
    if total <= N_FRAMES:
        return list(range(total)) + [total - 1] * (N_FRAMES - total)
    L = total // N_FRAMES
    starts = np.arange(N_FRAMES) * L
    ends = np.minimum(starts + L, total)
    if training:
        return [int(rng.integers(s, e)) if e > s else int(s) for s, e in zip(starts, ends)]
    return ((starts + ends) // 2).astype(int).tolist()


def crop_batch(frames):
    """Igual que el TSN V4: lado corto SHORT_SIDE (bilineal con antialias) + recorte central CROP.
    uint8 [T,H,W,3] -> uint8 [T,CROP,CROP,3]."""
    x = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2)
    if min(x.shape[-2:]) != SHORT_SIDE:
        x = TF.resize(x, SHORT_SIDE, interpolation=InterpolationMode.BILINEAR, antialias=True)
    return TF.center_crop(x, [CROP, CROP]).permute(0, 2, 3, 1).contiguous().numpy()


def crop_frame(frame):
    return crop_batch([frame])[0]


def to_tensor(frames, recortados=False):
    """Frames RGB uint8 -> tensor normalizado (ImageNet) [C,T,H,W]. `recortados=True` si ya vienen en CROP x CROP."""
    x = torch.from_numpy(np.stack(frames) if recortados else crop_batch(frames)).permute(0, 3, 1, 2).float().div_(255)
    x = TF.normalize(x, MEAN, STD)
    return x.permute(1, 0, 2, 3).contiguous()


# ---------- Banco de frames ----------
def candidate_matrix(total):
    """[N, K] índices de frame: K candidatos repartidos dentro de cada segmento. La columna K//2 es el frame
    central del segmento, el mismo que usa la evaluación del TSN V4."""
    K = CANDIDATOS_POR_SEGMENTO
    if total <= N_FRAMES:
        return np.repeat(np.array(segment_indices(total, False))[:, None], K, axis=1)
    L = total // N_FRAMES
    s = np.arange(N_FRAMES)[:, None] * L
    e = np.minimum(s + L, total)
    m = s + ((np.arange(K)[None, :] + 0.5) * (e - s) / K).astype(int)
    m[:, K // 2] = ((s + e) // 2)[:, 0]
    return np.clip(m, 0, total - 1)


def banco_nombre(key):
    return hashlib.sha1(s3_key_efectiva(key).encode()).hexdigest()[:20] + '.npz'


def banco_path(key):
    return BANCO_DIR / banco_nombre(key)


def banco_log(msg):
    with open(BANCO_DIR / 'extraccion_log.txt', 'a', encoding='utf-8') as f:
        f.write(f'{datetime.now():%Y-%m-%d %H:%M:%S} | {msg}\n')


def extraer_video(key, en_s3=frozenset()):
    """Descarga el video a un archivo temporal, extrae N x K frames ya recortados a CROP x CROP, los guarda
    comprimidos en JPEG en un .npz y borra el video. Si el banco ya existe (local o en S3), lo reutiliza."""
    row = {'s3_key': key, 'ok': False, 'motivo': '', 'origen': '', 'frames': np.nan, 'fps': np.nan,
           'ancho': np.nan, 'alto': np.nan, 'duracion_s': np.nan, 'kb_banco': np.nan}
    dest = banco_path(key)
    try:
        if dest.exists():
            row.update(ok=True, origen='local', kb_banco=dest.stat().st_size / 1024); return row
        if dest.name in en_s3:
            part = dest.with_name(dest.stem + f'.{threading.get_ident()}.part')
            s3().download_file(S3_BUCKET, BANCO_S3_PREFIX + dest.name, str(part)); os.replace(part, dest)
            row.update(ok=True, origen='s3', kb_banco=dest.stat().st_size / 1024); return row
        src = s3_key_efectiva(key)
        tmp = BANCO_TMP / f'{dest.stem}_{threading.get_ident()}{Path(src).suffix.lower() or ".mp4"}'
        try:
            s3().download_file(S3_BUCKET, src, str(tmp))
            if tmp.stat().st_size < 1024: raise RuntimeError('archivo vacío o casi vacío')
            n, fps, w, h = video_meta(tmp)
            row.update(frames=n, fps=fps, ancho=w, alto=h, duracion_s=n / fps)
            if n < 1: raise RuntimeError('video sin frames')
            cand = candidate_matrix(n)
            uniq, inv = np.unique(cand.ravel(), return_inverse=True)
            crops = read_frames(tmp, uniq.tolist(), fn=crop_frame)
        finally:
            tmp.unlink(missing_ok=True)
        jpgs = [cv2.imencode('.jpg', cv2.cvtColor(c, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, CALIDAD_JPEG])[1].ravel()
                for c in crops]
        part = dest.with_name(dest.stem + f'.{threading.get_ident()}.part.npz')
        np.savez(part, data=np.concatenate(jpgs), offsets=np.cumsum([0] + [len(j) for j in jpgs]),
                 mapa=inv.reshape(cand.shape), indices=cand, meta=np.array([n, fps, w, h], dtype=float))
        os.replace(part, dest)
        if COPIAR_BANCO_S3:
            s3().upload_file(str(dest), S3_BUCKET, BANCO_S3_PREFIX + dest.name)
        row.update(ok=True, origen='video', kb_banco=dest.stat().st_size / 1024)
    except ClientError as exc:
        row['motivo'] = f"S3 {exc.response['Error'].get('Code', '')}: no existe o sin permiso"
    except Exception as exc:
        row['motivo'] = f'{type(exc).__name__}: {str(exc)[:150]}'
    return row


def clip_desde_banco(key, training, rng=None):
    """Lee del banco los 16 frames: en entrenamiento uno al azar entre los K candidatos de cada segmento;
    en evaluación el central. Devuelve (clip [C,T,H,W], índices de frame, meta [frames, fps, ancho, alto])."""
    with np.load(banco_path(key)) as z:
        data, off, mapa, idx, meta = z['data'], z['offsets'], z['mapa'], z['indices'], z['meta']
    filas = np.arange(mapa.shape[0])
    js = rng.integers(0, mapa.shape[1], mapa.shape[0]) if training else np.full(mapa.shape[0], mapa.shape[1] // 2)
    frames = [cv2.cvtColor(cv2.imdecode(data[off[i]:off[i + 1]], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB) for i in mapa[filas, js]]
    return to_tensor(frames, recortados=True), idx[filas, js].tolist(), meta


def clip_eval(key):
    """Clip de evaluación (16 frames centrales) desde el banco o, si no existe, desde el video."""
    if USAR_BANCO_FRAMES and banco_path(key).exists():
        clip, idx, meta = clip_desde_banco(key, False)
        return clip, idx, int(meta[0]), float(meta[1])
    path = get_local_video(key); total, fps, _, _ = video_meta(path)
    idx = segment_indices(total, False)
    return to_tensor(read_frames(path, idx)), idx, total, fps


def denorm(x):
    """[C,T,H,W] normalizado -> [T,H,W,C] en [0,1] para graficar."""
    x = x * torch.tensor(STD).view(3, 1, 1, 1) + torch.tensor(MEAN).view(3, 1, 1, 1)
    return x.clamp(0, 1).permute(1, 2, 3, 0).numpy()


def registrar_fallo(key, exc, contexto):
    """Registro compartido entre procesos (los workers del DataLoader no comparten listas de Python)."""
    try:
        with open(FALLOS_CSV, 'a', encoding='utf-8') as f:
            f.write(f'{datetime.now():%Y-%m-%d %H:%M:%S},{contexto},"{key}","{type(exc).__name__}: {str(exc)[:150]}"\n')
    except OSError:
        pass


class VideoDataset(Dataset):
    """Devuelve (clip [C,T,H,W], etiqueta, s3_key) o None si el video falla (se omite del lote)."""
    def __init__(self, frame, training, contexto='datos'):
        self.frame, self.training, self.contexto = frame.reset_index(drop=True), training, contexto

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, ix):
        row = self.frame.iloc[ix]
        try:
            rng = np.random.default_rng(random.getrandbits(32))
            if USAR_BANCO_FRAMES and banco_path(row.s3_key).exists():
                return clip_desde_banco(row.s3_key, self.training, rng)[0], torch.tensor(float(row.y)), row.s3_key
            path = get_local_video(row.s3_key)
            total = video_meta(path)[0]
            if total < 1: raise RuntimeError('video sin frames')
            return to_tensor(read_frames(path, segment_indices(total, self.training, rng))), torch.tensor(float(row.y)), row.s3_key
        except Exception as exc:
            registrar_fallo(row.s3_key, exc, self.contexto)
            return None


def collate_skip(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    x, y, k = zip(*batch)
    return torch.stack(x), torch.stack(y), list(k)


def seed_worker(_):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)


def make_loader(frame, training, contexto='datos', batch_size=None):
    g = torch.Generator(); g.manual_seed(SEED)
    nw = NUM_WORKERS
    return DataLoader(VideoDataset(frame, training, contexto), batch_size=batch_size or BATCH_SIZE, shuffle=training,
                      num_workers=nw, pin_memory=AMP, worker_init_fn=seed_worker, generator=g, collate_fn=collate_skip,
                      persistent_workers=False, timeout=LOADER_TIMEOUT_S if nw > 0 else 0)


def verificar_video(key):
    """Descarga (caché), abre y lee exactamente los 16 frames de evaluación. Devuelve una fila de diagnóstico."""
    row = {'s3_key': key, 'ok': False, 'motivo': '', 'frames': np.nan, 'fps': np.nan, 'ancho': np.nan, 'alto': np.nan,
           'duracion_s': np.nan}
    try:
        path = get_local_video(key)
        if path.stat().st_size < 1024:
            raise RuntimeError('archivo vacío o casi vacío')
        n, fps, w, h = video_meta(path)
        row.update(frames=n, fps=fps, ancho=w, alto=h, duracion_s=n / fps if fps else np.nan)
        if n < 1: raise RuntimeError('video sin frames')
        fr = read_frames(path, segment_indices(n, False))
        if any(f is None or f.size == 0 for f in fr): raise RuntimeError('frames vacíos')
        row['ok'] = True
    except ClientError as exc:
        row['motivo'] = f"S3 {exc.response['Error'].get('Code', '')}: no existe o sin permiso"
    except Exception as exc:
        row['motivo'] = f'{type(exc).__name__}: {str(exc)[:150]}'
    return row

In [ ]:
# ================== 3.3 Modelos ==================
def build_x3d(dropout):
    """X3D-M preentrenado en Kinetics-400 con cabeza binaria (un logit)."""
    from pytorchvideo.models.hub.x3d import x3d_m
    net = x3d_m(pretrained=True)
    head = net.blocks[-1]
    head.pool.pool = nn.AdaptiveAvgPool3d(1)      # el pooling original es fijo para 16x7x7
    head.dropout, head.activation = nn.Dropout(dropout), None
    head.proj = nn.Linear(head.proj.in_features, 1)
    nn.init.normal_(head.proj.weight, std=0.01); nn.init.zeros_(head.proj.bias)
    return net


def set_phase(net, phase, n_etapas):
    """Fase 1: solo la neurona de salida. Fase 2: cabeza + las últimas `n_etapas` etapas (5 = red completa)."""
    net.requires_grad_(False)
    modules = [net.blocks[-1].proj] if phase == 1 else list(net.blocks[-1 - n_etapas:])
    for m in modules: m.requires_grad_(True)


def freeze_bn_stats(net):
    for m in net.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm): m.eval()


class BaselineTSN_V4(nn.Module):
    """Copia exacta de modelamiento_version4/scripts/models.py (mismos nombres de pesos)."""
    def __init__(self, num_classes=1, dropout=0.4):
        super().__init__()
        self.backbone = resnet50(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        B, C, T, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4).contiguous().view(B * T, C, H, W)
        logits = self.fc(self.dropout(self.backbone(x)))
        return logits.view(B, T, -1).mean(dim=1)


def cargar_checkpoint(path):
    """El TSN se guardó con pickle.dump (no con torch.save): se intenta torch.load y luego pickle, siempre en CPU."""
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except Exception:
        import torch.storage as ts
        original = ts._load_from_bytes
        ts._load_from_bytes = lambda b: torch.load(io.BytesIO(b), map_location='cpu', weights_only=False)
        try:
            with open(path, 'rb') as f:
                return pickle.load(f)
        finally:
            ts._load_from_bytes = original


def load_tsn(key):
    local = MODELS_DIR / ('tsn_' + key.replace('/', '_'))
    if not local.exists():
        s3().download_file(S3_BUCKET, key, str(local))
    ck = cargar_checkpoint(local)
    sd = ck.get('model_state_dict', ck) if isinstance(ck, dict) else ck.state_dict()
    sd = {k[7:] if k.startswith('module.') else k: v.float().cpu() for k, v in sd.items() if torch.is_tensor(v)}
    tsn = BaselineTSN_V4(dropout=0.4)
    tsn.load_state_dict(sd, strict=True)
    info = {k: ck[k] for k in ('epoch', 'best_val_auc') if isinstance(ck, dict) and k in ck}
    hist = ck.get('history') if isinstance(ck, dict) else None
    return tsn.to(DEVICE).eval(), info, hist

In [ ]:
# ================== 3.4 Entrenamiento con controles anticaídas ==================
class SmoothBCE(nn.Module):
    """BCE con logits y label smoothing (igual que el TSN V4): y' = y(1-s) + 0.5 s."""
    def __init__(self, smoothing=0.0):
        super().__init__(); self.s = smoothing

    def forward(self, z, y):
        return F.binary_cross_entropy_with_logits(z, y * (1 - self.s) + 0.5 * self.s)


class EarlyStoppingDoble:
    """Igual que el TSN V4: se detiene si val_loss o val_AUC no mejoran durante `patience` épocas."""
    def __init__(self, patience, min_delta=1e-3):
        self.patience, self.min_delta = patience, min_delta
        self.best_loss = self.best_auc = None; self.c_loss = self.c_auc = 0

    def step(self, loss, auc):
        if self.best_loss is None or loss < self.best_loss - self.min_delta: self.best_loss, self.c_loss = loss, 0
        else: self.c_loss += 1
        if self.best_auc is None or auc > self.best_auc + self.min_delta: self.best_auc, self.c_auc = auc, 0
        else: self.c_auc += 1
        return self.c_loss >= self.patience or self.c_auc >= self.patience


def log_line(run_dir, msg):
    line = f'{datetime.now():%Y-%m-%d %H:%M:%S} | {msg}'
    with open(run_dir / 'train_log.txt', 'a', encoding='utf-8') as f: f.write(line + '\n')
    print(line)


def atomic_save(payload, dest):
    tmp = dest.with_suffix('.tmp'); torch.save(payload, tmp); os.replace(tmp, dest)


def respaldar_s3(run_dir, names):
    if not RESPALDO_S3: return
    for n in names:
        f = run_dir / n
        if f.exists():
            try:
                s3().upload_file(str(f), S3_BUCKET, f'{S3_RESULT_PREFIX}{run_dir.parent.name}/{run_dir.name}/{n}')
            except Exception as exc:
                log_line(run_dir, f'AVISO respaldo S3 fallido ({n}): {exc}')


def asegurar_disco(run_dir):
    if espacio_libre_gb() < MIN_DISCO_LIBRE_GB:
        trim_cache(min_free_gb=MIN_DISCO_LIBRE_GB + 5)
        log_line(run_dir, f'Disco bajo: caché recortada; libre ahora {espacio_libre_gb():.1f} GB')
    if espacio_libre_gb() < min(3, MIN_DISCO_LIBRE_GB):
        raise RuntimeError('Menos de 3 GB libres: liberar espacio (sección 1, problemas frecuentes) y reanudar.')


@torch.inference_mode()
def predict(net, loader, desc='Inferencia'):
    net.eval(); rows = []
    for batch in tqdm(loader, desc=desc, leave=False):
        if batch is None: continue
        x, y, keys = batch
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=AMP):
            z = net(x.to(DEVICE, non_blocking=True)).reshape(-1).float()
        rows += list(zip(keys, y.int().tolist(), z.cpu().tolist()))
    out = pd.DataFrame(rows, columns=['s3_key', 'y_true', 'logit'])
    out['p'] = sigmoid(out.logit); out['y_pred'] = (out.p >= THRESHOLD).astype(int)
    return out


def run_epoch(net, loader, opt, scaler, loss_fn, phase, run_dir, desc):
    """Una época de entrenamiento tolerante a fallos de lectura, OOM y pérdidas no finitas."""
    net.train()
    if phase == 1 or FREEZE_BN: freeze_bn_stats(net)
    opt.zero_grad(set_to_none=True)
    ys, zs, losses = [], [], []
    cnt = dict(lotes_vacios=0, oom=0, loss_no_finita=0, pasos=0)
    pending = 0
    for batch in tqdm(loader, desc=desc, leave=False):
        if batch is None:
            cnt['lotes_vacios'] += 1; continue
        x, y, _ = batch
        try:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=AMP):
                z = net(x).reshape(-1).float()
            loss = loss_fn(z, y)
            if not torch.isfinite(loss):
                cnt['loss_no_finita'] += 1; opt.zero_grad(set_to_none=True); pending = 0; continue
            scaler.scale(loss / ACCUM_STEPS).backward()
        except torch.cuda.OutOfMemoryError:
            cnt['oom'] += 1
            opt.zero_grad(set_to_none=True); pending = 0
            del x, y
            torch.cuda.empty_cache()
            if cnt['oom'] > MAX_OOM_POR_EPOCA:
                raise RuntimeError(f'OOM repetido ({cnt["oom"]} veces): bajar BATCH_SIZE y subir ACCUM_STEPS.')
            continue
        pending += 1
        if pending == ACCUM_STEPS:
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); pending = 0; cnt['pasos'] += 1
        ys += y.int().tolist(); zs += z.detach().cpu().tolist(); losses.append(loss.item() * len(y))
    if pending:
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); cnt['pasos'] += 1
    if not ys:
        raise RuntimeError('La época no procesó ningún lote: revisar fallos_lectura.csv.')
    if cnt['oom'] or cnt['loss_no_finita'] or cnt['lotes_vacios']:
        log_line(run_dir, f'{desc}: incidencias {cnt}')
    return ys, zs, sum(losses) / len(ys), cnt


def build_optimizer(net, cfg, phase):
    params = [p for p in net.parameters() if p.requires_grad]
    if phase == 1:
        return torch.optim.AdamW(params, lr=cfg['lr1'], weight_decay=WEIGHT_DECAY), None
    opt = torch.optim.AdamW(params, lr=cfg['lr2'], weight_decay=WEIGHT_DECAY)
    if cfg['scheduler'] == 'coseno':
        return opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['ep2'], eta_min=cfg['lr2'] * 1e-2)
    return opt, torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)


def train_version(v):
    """Entrena la versión `v` de VERSIONES con reanudación, reintentos por época y registro en disco."""
    cfg = VERSIONES[v]
    run_dir = RUNS_DIR / v; run_dir.mkdir(parents=True, exist_ok=True)
    sig = dict(cfg=cfg, n_frames=N_FRAMES, crop=CROP, split=SPLIT_HASH, seed=SEED)
    total_ep = cfg['ep1'] + cfg['ep2']
    net = build_x3d(cfg['dropout']).to(DEVICE)
    loss_fn = SmoothBCE(cfg['label_smoothing'])
    scaler = torch.amp.GradScaler(DEVICE, enabled=AMP)
    hist, start, best, best_ep, stopped, pending = [], 0, -np.inf, 0, False, None
    es = EarlyStoppingDoble(cfg['early_stopping']) if cfg['early_stopping'] else None
    last_f, best_f = run_dir / 'last.pt', run_dir / 'best.pt'

    if REANUDAR and last_f.exists():
        st = torch.load(last_f, map_location='cpu', weights_only=False)
        if st['sig'] != sig:
            raise ValueError(f'{v}: last.pt es de otra configuración. Borrar {run_dir} o restaurar los parámetros.')
        net.load_state_dict(st['model']); scaler.load_state_dict(st['scaler'])
        hist, start, best, best_ep, stopped = st['hist'], st['epoch'], st['best'], st['best_ep'], st['stopped']
        if es and st.get('es'): es.__dict__.update(st['es'])
        pending = (st['phase'], st['opt'], st['sched'])
        if not stopped and start < total_ep:
            log_line(run_dir, f'{v}: reanudando en la época {start + 1}/{total_ep} (mejor época {best_ep})')
    if stopped or start >= total_ep:
        log_line(run_dir, f'{v}: entrenamiento ya terminado (mejor época {best_ep}).')
        return pd.DataFrame(hist)

    train_loader = make_loader(df_train, True, f'train-{v}')
    val_loader = make_loader(df_val, False, f'val-{v}')
    cur, opt, sched = None, None, None
    log_line(run_dir, f'{v} ({cfg["nombre"]}): {len(df_train)} videos train, {len(df_val)} val, {total_ep} épocas')

    for ep in range(start, total_ep):
        phase = 1 if ep < cfg['ep1'] else 2
        if phase != cur:
            set_phase(net, phase, cfg['etapas_fase2']); opt, sched = build_optimizer(net, cfg, phase); cur = phase
            if pending and pending[0] == phase:
                opt.load_state_dict(pending[1])
                if sched is not None and pending[2]: sched.load_state_dict(pending[2])
            pending = None
        asegurar_disco(run_dir)
        snap = (copy.deepcopy(net.state_dict()), copy.deepcopy(opt.state_dict()), copy.deepcopy(scaler.state_dict()))
        for intento in range(1, MAX_REINTENTOS_EPOCA + 2):
            try:
                if AMP: torch.cuda.reset_peak_memory_stats()
                t0 = time.perf_counter()
                ys, zs, tr_loss, cnt = run_epoch(net, train_loader, opt, scaler, loss_fn, phase, run_dir,
                                                 f'{v} época {ep + 1}/{total_ep} (fase {phase})')
                t_train = time.perf_counter() - t0
                vram = torch.cuda.max_memory_allocated() / 1024**2 if AMP else np.nan
                t0 = time.perf_counter(); pv = predict(net, val_loader, f'{v} validación'); t_val = time.perf_counter() - t0
                break
            except (RuntimeError, OSError, ConnectionError) as exc:
                if 'OOM repetido' in str(exc) or 'Menos de 3 GB' in str(exc) or intento > MAX_REINTENTOS_EPOCA:
                    log_line(run_dir, f'{v}: ERROR definitivo en la época {ep + 1}: {exc}'); raise
                log_line(run_dir, f'{v}: fallo en la época {ep + 1} (intento {intento}): {type(exc).__name__}: {str(exc)[:200]}. '
                                  'Se restaura el estado y se reintenta.')
                net.load_state_dict(snap[0]); opt.load_state_dict(snap[1]); scaler.load_state_dict(snap[2])
                if AMP: torch.cuda.empty_cache()
                train_loader = make_loader(df_train, True, f'train-{v}')
                val_loader = make_loader(df_val, False, f'val-{v}')
                time.sleep(10)

        lr = opt.param_groups[0]['lr']
        tm, vm = metrics(ys, sigmoid(zs)), metrics(pv.y_true, pv.p)
        val_loss = bce(pv.y_true, pv.logit)
        if sched is not None:
            sched.step(val_loss) if cfg['scheduler'] == 'plateau' else sched.step()
        row = {'epoch': ep + 1, 'fase': phase, 'lr': lr, 'train_loss': tr_loss, 'val_loss': val_loss,
               **{f'train_{k}': tm[k] for k in METRICAS}, **{f'val_{k}': vm[k] for k in METRICAS},
               'seg_entrenamiento': t_train, 'seg_validacion': t_val, 'vram_mb': vram, 'n_val': len(pv), **cnt}
        hist.append(row)
        score = vm['auc'] if np.isfinite(vm['auc']) else -val_loss
        improved = score > best
        if improved: best, best_ep = score, ep + 1
        if es and phase == 2 and es.step(val_loss, vm['auc']):
            stopped = True
        payload = dict(model=net.state_dict(), opt=opt.state_dict(), sched=sched.state_dict() if sched else None,
                       scaler=scaler.state_dict(), phase=phase, epoch=ep + 1, best=best, best_ep=best_ep,
                       stopped=stopped, es=es.__dict__ if es else None, hist=hist, sig=sig)
        atomic_save(payload, last_f)
        if improved: atomic_save(payload, best_f)
        pd.DataFrame(hist).to_csv(run_dir / 'history.csv', index=False)
        respaldar_s3(run_dir, ['history.csv', 'train_log.txt'] + (['best.pt'] if improved else []))
        log_line(run_dir, f"{v} época {ep + 1}/{total_ep} (fase {phase}) | loss {tr_loss:.4f}/{val_loss:.4f} | "
                          f"AUC val {vm['auc']:.4f} | F1 val {vm['f1']:.4f} | {t_train / 60:.1f} min | VRAM {vram:.0f} MB"
                          + ('  <- mejor' if improved else ''))
        if stopped:
            log_line(run_dir, f'{v}: early stopping en la época {ep + 1} (mejor época {best_ep}).'); break
    return pd.DataFrame(hist)

In [ ]:
# ================== 3.5 Métricas y pruebas estadísticas ==================
METRICAS = ['auc', 'f1', 'precision', 'recall', 'accuracy']
NOMBRE = {0: 'No violento', 1: 'Violento'}
sigmoid = lambda z: 1 / (1 + np.exp(-np.asarray(z, float)))


def metrics(y, p, thr=None):
    thr = THRESHOLD if thr is None else thr
    y, p = np.asarray(y, int), np.asarray(p, float)
    yhat = (p >= thr).astype(int)
    return {'auc': roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
            'auprc': average_precision_score(y, p) if y.any() else np.nan,
            'f1': f1_score(y, yhat, zero_division=0), 'precision': precision_score(y, yhat, zero_division=0),
            'recall': recall_score(y, yhat, zero_division=0), 'accuracy': accuracy_score(y, yhat)}


def bce(y, z):
    return float(F.binary_cross_entropy_with_logits(torch.tensor(np.asarray(z, float)), torch.tensor(np.asarray(y, float))))


def bootstrap_table(y, p, n=None, seed=None):
    n, seed = n or N_BOOTSTRAP, SEED if seed is None else seed
    y, p = np.asarray(y, int), np.asarray(p, float)
    rng = np.random.default_rng(seed)
    strata = [np.where(y == c)[0] for c in (0, 1) if (y == c).any()]
    reps = pd.DataFrame([metrics(y[i], p[i]) for i in
                         (np.concatenate([rng.choice(s, len(s)) for s in strata]) for _ in range(n))])
    point = metrics(y, p)
    keys = ['auc', 'auprc'] + METRICAS[1:]
    return pd.DataFrame({'valor': [point[k] for k in keys], 'ic95_inf': reps[keys].quantile(0.025).values,
                         'ic95_sup': reps[keys].quantile(0.975).values}, index=keys).round(4)


def fmt_ci(t):
    return {m: f"{t.loc[m, 'valor']:.3f} [{t.loc[m, 'ic95_inf']:.3f}, {t.loc[m, 'ic95_sup']:.3f}]" for m in t.index}


def delong_test(y, p1, p2):
    """DeLong et al. (1988), implementación rápida de Sun & Xu (2014)."""
    y = np.asarray(y, int); pos = y == 1; m, n = pos.sum(), (~pos).sum()
    aucs, v10, v01 = [], [], []
    for p in (np.asarray(p1, float), np.asarray(p2, float)):
        tz, tx, ty = stats.rankdata(p), stats.rankdata(p[pos]), stats.rankdata(p[~pos])
        v10.append((tz[pos] - tx) / n); v01.append(1 - (tz[~pos] - ty) / m); aucs.append(v10[-1].mean())
    s = np.cov(np.vstack(v10)) / m + np.cov(np.vstack(v01)) / n
    var = s[0, 0] + s[1, 1] - 2 * s[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else 0.0
    return aucs[0], aucs[1], z, 2 * stats.norm.sf(abs(z))


def mcnemar_exact(y, yhat1, yhat2):
    ok1, ok2 = np.asarray(yhat1) == np.asarray(y), np.asarray(yhat2) == np.asarray(y)
    b, c = int((ok1 & ~ok2).sum()), int((~ok1 & ok2).sum())
    return b, c, (stats.binomtest(b, b + c, 0.5).pvalue if b + c else 1.0)


def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); adj = np.empty_like(p)
    adj[order] = np.minimum(1, np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]))
    return adj


@torch.inference_mode()
def bench(net, batch, iters):
    net.eval(); x = torch.randn(batch, 3, N_FRAMES, CROP, CROP, device=DEVICE)
    if AMP: torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    for i in range(iters + 2):
        if i == 2:
            if AMP: torch.cuda.synchronize()
            t0 = time.perf_counter()
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=AMP):
            net(x)
    if AMP: torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters, (torch.cuda.max_memory_allocated() / 1024**2 if AMP else np.nan)


def gflops(net):
    try:
        from fvcore.nn import FlopCountAnalysis
        fl = FlopCountAnalysis(net.eval(), torch.zeros(1, 3, N_FRAMES, CROP, CROP, device=DEVICE))
        fl.unsupported_ops_warnings(False); fl.uncalled_modules_warnings(False)
        return fl.total() / 1e9
    except Exception as exc:
        print('FLOPs no medibles:', type(exc).__name__); return np.nan


def costos_modelo(net, iters=None):
    iters = iters or (3 if MODO == 'PRUEBA' else 30)
    lat, _ = bench(net, 1, iters); tb, vram = bench(net, BATCH_SIZE, iters)
    return {'parámetros (M)': sum(p.numel() for p in net.parameters()) / 1e6, 'GFLOPs por video': gflops(net),
            'VRAM inferencia (MB)': vram, 'latencia por video (ms)': lat * 1e3, 'throughput (videos/s)': BATCH_SIZE / tb}

In [ ]:
# ================== 3.6 Gráficos, Grad-CAM y galería ==================
TITULOS = {'loss': 'Pérdida BCE', 'auc': 'AUC-ROC', 'f1': 'F1 (τ=0.5)', 'precision': 'Precisión (τ=0.5)',
           'recall': 'Recall (τ=0.5)', 'accuracy': 'Accuracy (τ=0.5)'}


def plot_history(h, title, ep1, path=None, ref=None, ref_name=None):
    """8 paneles: métricas de train y val por época, tiempo y VRAM."""
    best_ep = int(h.loc[h.val_auc.fillna(-1).idxmax(), 'epoch'])
    fig, axes = plt.subplots(2, 4, figsize=(20, 8.5))
    for ax, (k, t) in zip(axes.flat, TITULOS.items()):
        ax.plot(h.epoch, h[f'train_{k}'], 'o-', ms=4, label='entrenamiento')
        ax.plot(h.epoch, h[f'val_{k}'], 'o-', ms=4, label='validación')
        if ref is not None and k in ref:
            ax.axhline(ref[k], color=PALETA[3], ls='--', lw=1.2, label=ref_name)
        ax.set(title=t, xlabel='época')
        if k != 'loss': ax.set_ylim(-0.02, 1.02)
    ax = axes.flat[6]
    ax.bar(h.epoch, h.seg_entrenamiento / 60, color=PALETA[1], label='entrenamiento')
    ax.bar(h.epoch, h.seg_validacion / 60, bottom=h.seg_entrenamiento / 60, color=PALETA[4], label='validación')
    ax.set(title='Tiempo por época (min)', xlabel='época'); ax.legend()
    ax = axes.flat[7]; ax.bar(h.epoch, h.vram_mb, color=PALETA[2]); ax.set(title='Pico de VRAM (MB)', xlabel='época')
    for ax in axes.flat:
        if ep1 < h.epoch.max(): ax.axvline(ep1 + 0.5, color='gray', ls=':', lw=1)
    for ax in axes.flat[:6]: ax.axvline(best_ep, color=PALETA[2], lw=0.8, alpha=0.7)
    axes.flat[0].legend(); axes.flat[1].legend()
    fig.suptitle(f'{title} | mejor época {best_ep} (AUC val {h.val_auc.max():.4f})')
    fig.tight_layout()
    if path: fig.savefig(path, dpi=150)
    plt.show()


def plot_confusion(ax, y, yhat, title):
    cm = confusion_matrix(y, yhat, labels=[0, 1])
    ax.imshow(cm, cmap='magma'); ax.grid(False)
    for (i, j), v in np.ndenumerate(cm): ax.text(j, i, v, ha='center', va='center', color='w', fontsize=14)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['No viol.', 'Viol.'], yticklabels=['No viol.', 'Viol.'],
           xlabel='Predicho', ylabel='Real', title=title)


def gradcam_x3d(net, clip):
    """Grad-CAM espaciotemporal sobre la salida de res5 (blocks[-2]). Devuelve (p, mapa [T,H,W] en [0,1])."""
    layer, acts, grads = net.blocks[-2], [], []
    def hook(_, __, out):
        acts.append(out); out.register_hook(grads.append)
    handle = layer.register_forward_hook(hook)
    req = [p.requires_grad for p in net.parameters()]
    try:
        net.eval(); net.requires_grad_(True); net.zero_grad(set_to_none=True)
        with torch.enable_grad():
            z = net(clip[None].to(DEVICE)).reshape(-1)[0]; z.backward()
        a, g = acts[0].detach().float(), grads[0].detach().float()
        cam = F.relu((a * g.mean(dim=(2, 3, 4), keepdim=True)).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=tuple(clip.shape[1:]), mode='trilinear', align_corners=False)[0, 0].cpu().numpy()
        return float(torch.sigmoid(z.detach())), (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    finally:
        handle.remove(); net.zero_grad(set_to_none=True)
        for p, r in zip(net.parameters(), req): p.requires_grad_(r)


def tipo_resultado(y, yhat):
    return {(1, 1): 'TP', (0, 0): 'TN', (0, 1): 'FP', (1, 0): 'FN'}[(int(y), int(yhat))]


def show_example(net, key, y_true, etiqueta, out_dir):
    """Figura (muestra + Grad-CAM + evidencia en el tiempo) y GIF (video original + mapa de calor)."""
    clip, idx, total, fps = clip_eval(key); vis = denorm(clip)
    p, cam = gradcam_x3d(net, clip)
    tipo = tipo_resultado(y_true, p >= THRESHOLD)
    title = f'{etiqueta} | Real: {NOMBRE[y_true]} | Predicción: {NOMBRE[int(p >= THRESHOLD)]} (p={p:.3f}, {tipo})\n{key}'
    ts = np.linspace(0, N_FRAMES - 1, 8).astype(int)
    fig = plt.figure(figsize=(18, 7.4)); gs = fig.add_gridspec(3, 8, height_ratios=[1, 1, 0.9])
    for j, t in enumerate(ts):
        a0, a1 = fig.add_subplot(gs[0, j]), fig.add_subplot(gs[1, j])
        a0.imshow(vis[t]); a1.imshow(vis[t]); a1.imshow(cam[t], cmap='inferno', alpha=0.5, vmin=0, vmax=1)
        a0.set_title(f'{idx[t] / fps:.1f} s', fontsize=9); a0.axis('off'); a1.axis('off')
        if j == 0: a1.set_title('Grad-CAM', fontsize=9, loc='left')
    ax = fig.add_subplot(gs[2, :])
    ax.plot(np.array(idx) / fps, cam.mean(axis=(1, 2)), 'o-', color=PALETA[0])
    ax.set(xlim=(0, total / fps), xlabel='tiempo en el video original (s)', ylabel='evidencia (Grad-CAM medio)')
    fig.suptitle(title, fontsize=10); fig.tight_layout()
    fig.savefig(out_dir / f'{etiqueta}_{Path(key).stem}.png', dpi=110); plt.show()

    path = get_local_video(key)                     # el video original solo se descarga para el GIF
    orig_idx = np.linspace(0, total - 1, min(FRAMES_ANIMACION, total)).round().astype(int)
    orig = read_frames(path, orig_idx, 180)
    nearest = [int(np.abs(np.array(idx) - i).argmin()) for i in orig_idx]
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.4))
    im0 = axes[0].imshow(orig[0]); im1 = axes[1].imshow(vis[nearest[0]])
    ov = axes[1].imshow(cam[nearest[0]], cmap='inferno', alpha=0.5, vmin=0, vmax=1)
    for a in axes: a.axis('off')
    axes[0].set_title('Video original', fontsize=9); axes[1].set_title(f'Muestra + Grad-CAM (p={p:.2f})', fontsize=9)
    txt = fig.suptitle('', fontsize=9)

    def update(f):
        im0.set_data(orig[f]); im1.set_data(vis[nearest[f]]); ov.set_data(cam[nearest[f]])
        txt.set_text(f'{etiqueta} | Real: {NOMBRE[y_true]} | t = {orig_idx[f] / fps:.1f} s | segmento {nearest[f] + 1}/{N_FRAMES}')
        return im0, im1, ov, txt

    gif = out_dir / f'{etiqueta}_{Path(key).stem}.gif'
    animation.FuncAnimation(fig, update, frames=len(orig), interval=150).save(gif, writer=animation.PillowWriter(fps=6), dpi=70)
    plt.close(fig)
    display(Image(filename=str(gif)))

## 4. Parámetros del notebook
**Única celda editable.** Los valores por defecto reproducen la partición, los frames y las dimensiones del TSN V4, y definen las tres versiones de X3D-Medium (sección 6).

- `MODO='PRUEBA'` usa pocos videos y una época por fase: sirve para validar que todo corre, no para reportar resultados.
- `MODO='COMPLETO'` usa la partición completa del TSN.
- Cambiar parámetros de una versión ya entrenada obliga a borrar su carpeta en `RUNS_DIR`: el notebook lo detecta y se detiene para no mezclar configuraciones.

In [ ]:
# --- Ejecución ---
MODO = 'COMPLETO'                     # 'PRUEBA' | 'COMPLETO'
SEED = 42                             # misma semilla del TSN V4
REANUDAR = True                       # continuar desde la última época guardada

# --- Datos (idénticos al TSN V4) ---
AWS_REGION = 'us-east-2'
S3_BUCKET = 'capstone-uai-cv'
MANIFEST_S3_KEY = 'df_manifest.csv'   # el manifest con el que se entrenó el TSN V4
LABEL_COL = None                      # None = regla del TSN: 'violent_label' si existe; si no, 'anomalía'
VAL_FRAC, TEST_FRAC = 0.15, 0.15      # 70 / 15 / 15, como el TSN V4
EXCLUIR_NO_PERMITIDAS = True          # True: quita exclude_from_training (deja de ser la misma muestra del TSN)
N_PRUEBA = {'train': 32, 'val': 12, 'test': 12}   # solo MODO='PRUEBA'
USAR_224 = False                      # True: extraer desde copias pre-convertidas a 224x224
PREFIJO_224 = 'videos_224/'
CARPETA_MUNICIPAL = 'VideosTest Pozo Almonte'
MUNICIPAL_MANIFEST_S3_KEY = ''        # opcional: CSV con columnas s3_key y etiqueta (0/1) de los videos municipales

# --- Banco de frames (sección 5) ---
USAR_BANCO_FRAMES = True              # entrenar y evaluar leyendo solo frames extraídos una vez (no los videos)
CANDIDATOS_POR_SEGMENTO = 8           # frames guardados por segmento: train sortea uno, val/test usan el central
CALIDAD_JPEG = 95
HILOS_EXTRACCION = 6
COPIAR_BANCO_S3 = True                # copia del banco en S3: no se vuelve a extraer al cambiar de instancia o disco

# --- Muestreo y preprocesamiento (igual al TSN V4) ---
N_FRAMES = 16
SHORT_SIDE, CROP = 256, 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]       # ImageNet

# --- Versiones de X3D-Medium (sección 6) ---
VERSIONES = {
    'V1': dict(nombre='Metodología: ajuste parcial (res4, res5)', ep1=5, lr1=1e-3, ep2=15, lr2=1e-5,
               etapas_fase2=2, scheduler='coseno', dropout=0.5, label_smoothing=0.0, early_stopping=None),
    'V2': dict(nombre='Ajuste fino completo (todas las etapas)', ep1=5, lr1=1e-3, ep2=15, lr2=1e-4,
               etapas_fase2=5, scheduler='coseno', dropout=0.5, label_smoothing=0.0, early_stopping=None),
    'V3': dict(nombre='Receta del TSN V4 (regularización)', ep1=5, lr1=1e-3, ep2=10, lr2=1e-5,
               etapas_fase2=2, scheduler='plateau', dropout=0.4, label_smoothing=0.1, early_stopping=3),
}

# --- Entrenamiento ---
BATCH_SIZE, ACCUM_STEPS = 4, 2        # batch efectivo 8
WEIGHT_DECAY = 1e-4
FREEZE_BN = True
GRAD_CLIP = 1.0
THRESHOLD = 0.5

# --- Controles anticaídas ---
NUM_WORKERS = 4
LOADER_TIMEOUT_S = 600                # un worker que no entrega un lote en 10 min se considera colgado
MAX_REINTENTOS_EPOCA = 3
MAX_OOM_POR_EPOCA = 5
MIN_DISCO_LIBRE_GB = 15
VERIFICAR_VIDEOS = True
HILOS_VERIFICACION = 8
EXCLUIR_SI_FALLA_N = 3                # videos que fallan N veces durante el entrenamiento se excluyen al relanzar
CACHE_MAX_GB = 170
MAX_FRAMES_SECUENCIAL = 1000
RESPALDO_S3 = True                    # True: sube history, log y best.pt a S3 en cada época
S3_RESULT_PREFIX = 'resultados/vigia_v3/'

# --- Baseline TSN ---
TSN_S3_KEY = 'modelamiento_version4/checkpoints/best_model.pt'

# --- Evaluación y visualización ---
N_BOOTSTRAP = 1000
N_EJEMPLOS_POR_TIPO = 2
FRAMES_ANIMACION = 32

# --- Rutas ---
BASE_DIR = Path.home() / 'capstone_vigia'
CACHE_DIR = BASE_DIR / 'cache_videos'
MODELS_DIR = BASE_DIR / 'modelos'
RUNS_DIR = BASE_DIR / 'runs_v3' / MODO.lower()

# --- Derivados (no editar) ---
if USAR_224: SHORT_SIDE = CROP
if MODO == 'PRUEBA':
    VERSIONES = {v: {**c, 'ep1': 1, 'ep2': 1} for v, c in VERSIONES.items()}
BANCO_NOMBRE = f'n{N_FRAMES}_k{CANDIDATOS_POR_SEGMENTO}_s{SHORT_SIDE}_c{CROP}_q{CALIDAD_JPEG}' + ('_v224' if USAR_224 else '')
BANCO_DIR = BASE_DIR / 'banco_frames' / BANCO_NOMBRE
BANCO_TMP = BASE_DIR / 'tmp_extraccion'
BANCO_S3_PREFIX = f'banco_frames/{BANCO_NOMBRE}/'
if USAR_BANCO_FRAMES:
    CACHE_MAX_GB = min(CACHE_MAX_GB, 20)   # con el banco, la caché de videos solo se usa en la galería
for d in (CACHE_DIR, MODELS_DIR, RUNS_DIR, BANCO_DIR, BANCO_TMP):
    d.mkdir(parents=True, exist_ok=True)
FALLOS_CSV = RUNS_DIR / 'fallos_lectura.csv'
cache_actual = sum(p.stat().st_size for p in CACHE_DIR.iterdir() if p.is_file())
disco_libre_gb = (shutil.disk_usage(CACHE_DIR).free + cache_actual) / 1024**3
CACHE_MAX_GB = min(CACHE_MAX_GB, max(1.0, disco_libre_gb - MIN_DISCO_LIBRE_GB))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

PALETA = ['#FF6F61', '#22D3EE', '#FBBF24', '#A78BFA', '#2DD4BF']
plt.style.use('dark_background')
plt.rcParams.update({'axes.prop_cycle': plt.cycler(color=PALETA), 'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.15, 'axes.spines.top': False, 'axes.spines.right': False})
print(f'Resultados en: {RUNS_DIR}\nCaché: hasta {CACHE_MAX_GB:.1f} GB (disco disponible {disco_libre_gb:.1f} GB)')

## 5. Carga de datos, misma partición del TSN y verificación de videos

**Misma muestra que el TSN.** El script `train.py` del TSN V4 divide `df_manifest.csv` así:
```python
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df[label])
train, val = train_test_split(train_val, test_size=0.15/(1-0.15), random_state=42, stratify=train_val[label])
```
Esa división es determinista: con el mismo archivo, en el mismo orden de filas, produce exactamente los mismos videos en cada partición. Aquí se ejecuta la misma instrucción, por lo que **X3D-M entrena con los mismos videos que el TSN y ambos se comparan en un test que ninguno vio**.

Dos verificaciones:
1. **Fechas en S3.** Si `df_manifest.csv` se modificó después de guardar el checkpoint del TSN, la partición podría no coincidir; la celda lo advierte.
2. **AUC de validación del TSN** (sección 13). El checkpoint guarda el AUC de validación con el que se seleccionó. Si al evaluar el TSN en la validación reproducida se obtiene un valor cercano, la partición coincide; si se acerca a 1.0, la "validación" contiene videos de su entrenamiento.

### Banco de frames
El modelo usa 16 frames por video, recortados a 224×224; los videos completos suman más de 180 GB. Por eso, con `USAR_BANCO_FRAMES=True`, cada video se procesa **una sola vez**:

1. Se descarga a un archivo temporal.
2. Se extraen $N\times K$ frames ($K$ = `CANDIDATOS_POR_SEGMENTO`) con el mismo preprocesamiento del TSN: lado corto 256 y recorte central 224×224.
3. Se guardan comprimidos en JPEG (calidad 95) en un archivo `.npz` por video, en `banco_frames/` de la EC2 y con copia en S3.
4. Se borra el video.

Los $K$ candidatos de cada segmento se reparten dentro de él, y uno de ellos es exactamente el frame central:

$$c_{k,j}=s_k+\Big\lfloor\frac{(j+\tfrac12)(e_k-s_k)}{K}\Big\rfloor,\quad j=0,\dots,K-1,\qquad c_{k,\lfloor K/2\rfloor}=\Big\lfloor\frac{s_k+e_k}{2}\Big\rfloor.$$

- **Entrenamiento:** en cada época se sortea un candidato por segmento.
- **Validación, test y municipal:** se usa el central, igual que el TSN.

Después de la extracción, el entrenamiento no decodifica video ni descarga nada de S3. La extracción es reanudable: si se interrumpe, al relanzar la celda solo procesa los videos que faltan, y si el banco ya existe en S3 lo copia en vez de volver a extraer.

### Videos corruptos
La extracción abre cada video y registra el motivo de cada falla (no existe en S3, no abre, sin frames, frames ilegibles) en `extraccion.csv`. Los videos con falla se excluyen de train, val y test **para ambos modelos**. Además, los videos que fallaron `EXCLUIR_SI_FALLA_N` veces durante entrenamientos anteriores (registrados en `fallos_lectura.csv`) se excluyen al relanzar. Con `USAR_BANCO_FRAMES=False` se usa la verificación sin banco: se leen los 16 frames de evaluación de cada video.

In [ ]:
# --- Acceso a S3 ---
try:
    print('Identidad IAM:', boto3.Session(region_name=AWS_REGION).client('sts').get_caller_identity()['Arn'])
    head_manifest = s3().head_object(Bucket=S3_BUCKET, Key=MANIFEST_S3_KEY)
except (ClientError, NoCredentialsError, BotoCoreError) as exc:
    raise RuntimeError('Sin acceso S3: revisar el rol IAM de la EC2, la región y el nombre del manifest.') from exc
try:
    head_tsn = s3().head_object(Bucket=S3_BUCKET, Key=TSN_S3_KEY)
    if head_manifest['LastModified'] > head_tsn['LastModified']:
        warnings.warn(f"{MANIFEST_S3_KEY} se modificó ({head_manifest['LastModified']:%Y-%m-%d %H:%M}) después del checkpoint del "
                      f"TSN ({head_tsn['LastModified']:%Y-%m-%d %H:%M}): la partición podría no coincidir. Ver la verificación de la sección 13.")
    else:
        print(f"Manifest ({head_manifest['LastModified']:%Y-%m-%d}) anterior al checkpoint del TSN ({head_tsn['LastModified']:%Y-%m-%d}): OK.")
except ClientError:
    warnings.warn(f'No se encontró {TSN_S3_KEY} en S3.')

# --- Manifest y partición idéntica al TSN V4 ---
manifest = pd.read_csv(io.BytesIO(s3().get_object(Bucket=S3_BUCKET, Key=MANIFEST_S3_KEY)['Body'].read()))
LABEL = LABEL_COL or ('violent_label' if 'violent_label' in manifest.columns else 'anomalía')
train_val_df, test_df = train_test_split(manifest, test_size=TEST_FRAC, random_state=SEED, stratify=manifest[LABEL])
train_df, val_df = train_test_split(train_val_df, test_size=VAL_FRAC / (1 - TEST_FRAC), random_state=SEED,
                                    stratify=train_val_df[LABEL])
print(f'Etiqueta: {LABEL} | manifest: {len(manifest)} videos')
for n, d in (('Train', train_df), ('Val', val_df), ('Test', test_df)):
    print(f'   {n}: {len(d)} videos ({int((d[LABEL] == 1).sum())} positivos)   <- comparar con el log del TSN V4')

col = lambda d, c, default: d[c].astype(str) if c in d else default
splits = pd.concat([d.assign(split=s) for s, d in (('train', train_df), ('val', val_df), ('test', test_df))])
splits = pd.DataFrame({'s3_key': splits.s3_key.astype(str), 'y': (pd.to_numeric(splits[LABEL]) == 1).astype(int),
                       'fuente': col(splits, 'dataset', splits.s3_key.str.split('/').str[1]),
                       'categoria': col(splits, 'video_sublabel_original', splits.s3_key.str.split('/').str[-2]),
                       'excluir': splits['exclude_from_training'].astype(str).str.lower().isin({'true', '1'})
                       if 'exclude_from_training' in splits else False,
                       'split': splits.split}).reset_index(drop=True)
splits['municipal'] = splits.s3_key.str.startswith(CARPETA_MUNICIPAL + '/') | splits.s3_key.str.contains('pozo almonte', case=False)
splits.to_csv(RUNS_DIR / 'particion_tsn_reproducida.csv', index=False)
if splits.municipal.any():
    print('\nAVISO: el manifest del TSN incluye videos municipales; por partición:', splits[splits.municipal].split.value_counts().to_dict())

display(pd.crosstab([splits.fuente, splits.categoria], [splits.split, splits.y]).rename_axis(columns=['partición', 'y']))
if EXCLUIR_NO_PERMITIDAS:
    print(f'Se quitan {int(splits.excluir.sum())} videos con exclude_from_training=True (deja de ser la misma muestra del TSN).')
    splits = splits[~splits.excluir]

# --- Videos municipales: del manifest municipal, del manifest principal o de la carpeta en S3 ---
VIOLENTAS = {'explosion', 'arrest', 'shoplifting', 'caraccident', 'roadaccidents', 'stealing', 'robbery', 'abuse', 'assault',
             'burglary', 'fighting', 'shooting', 'vandalism', 'riot', 'fight', 'figth', 'arson', 'violento', 'violencia', 'pelea'}
NORMALES = {'nonfight', 'nofight', 'normal', 'normal_video', 'no_violento', 'noviolento', 'no_violencia', 'sin_violencia'}
norm_txt = lambda t: str(t).lower().strip().replace('-', '_').replace(' ', '_')


def etiqueta_desde_ruta(key):
    for part in map(norm_txt, key.split('/')[1:-1]):
        if part in NORMALES: return 0
        if part in VIOLENTAS: return 1
    return np.nan


if MUNICIPAL_MANIFEST_S3_KEY:
    mm = pd.read_csv(io.BytesIO(s3().get_object(Bucket=S3_BUCKET, Key=MUNICIPAL_MANIFEST_S3_KEY)['Body'].read()))
    df_muni = pd.DataFrame({'s3_key': mm.s3_key.astype(str), 'y': pd.to_numeric(mm['etiqueta'], errors='coerce')})
elif splits.municipal.any():
    df_muni = splits.loc[splits.municipal, ['s3_key', 'y', 'split']].copy()
else:
    keys_m = [k for k in s3_sizes([CARPETA_MUNICIPAL + '/']) if k.lower().endswith(VIDEO_EXT)]
    df_muni = pd.DataFrame({'s3_key': keys_m, 'y': [etiqueta_desde_ruta(k) for k in keys_m]})
df_muni = df_muni.reset_index(drop=True)
print(f'\nVideos municipales: {len(df_muni)} ({int(df_muni.y.notna().sum())} con etiqueta)')

# --- Submuestra de prueba ---
if MODO == 'PRUEBA':
    splits = pd.concat([d.sample(n=min(len(d), N_PRUEBA[s] // 2), random_state=SEED)
                        for (s, _), d in splits.groupby(['split', 'y'])], ignore_index=True)
    df_muni = df_muni.head(4)

In [ ]:
# --- Banco de frames (o verificación de videos sin banco) ---
claves = sorted(set(splits.s3_key) | set(df_muni.s3_key))
if USAR_BANCO_FRAMES:
    trim_cache()                                          # aplica el nuevo límite: libera los videos completos ya descargados
    for f in BANCO_TMP.iterdir(): f.unlink(missing_ok=True)
    for f in BANCO_DIR.glob('*.part*'): f.unlink(missing_ok=True)
    VER_CSV = BANCO_DIR / 'extraccion.csv'
    prev = pd.read_csv(VER_CSV) if VER_CSV.exists() else pd.DataFrame(columns=['s3_key', 'ok', 'motivo'])
    pendientes = [k for k in claves if not banco_path(k).exists()]
    en_s3 = frozenset(Path(k).name for k in s3_sizes([BANCO_S3_PREFIX])) if COPIAR_BANCO_S3 and pendientes else frozenset()
    print(f'Banco {BANCO_NOMBRE}: {len(claves) - len(pendientes)} videos listos | {len(pendientes)} pendientes '
          f'({sum(banco_nombre(k) in en_s3 for k in pendientes)} se copian desde S3) | disco libre {espacio_libre_gb():.1f} GB')
    filas, t0 = [], time.time()
    if pendientes:
        banco_log(f'Inicio: {len(pendientes)} pendientes')
        with ThreadPoolExecutor(HILOS_EXTRACCION) as ex:
            futs = [ex.submit(extraer_video, k, en_s3) for k in pendientes]
            for n_ok, fut in enumerate(tqdm(as_completed(futs), total=len(futs), desc='Extrayendo frames'), start=1):
                filas.append(fut.result())
                if n_ok % 200 == 0 or n_ok == len(futs):
                    hechos = pd.DataFrame(filas)
                    pd.concat([prev[~prev.s3_key.isin(hechos.s3_key)], hechos], ignore_index=True).to_csv(VER_CSV, index=False)
                    rate = n_ok / (time.time() - t0)
                    banco_log(f'{n_ok}/{len(futs)} | fallas {int((~hechos.ok).sum())} | {rate * 3600:.0f} videos/h | '
                              f'faltan ~{(len(futs) - n_ok) / rate / 3600:.1f} h | disco libre {espacio_libre_gb():.1f} GB')
    ver = pd.concat([prev[~prev.s3_key.isin([r['s3_key'] for r in filas])], pd.DataFrame(filas)], ignore_index=True)
    ver.to_csv(VER_CSV, index=False)
    ok_keys = {k for k in claves if banco_path(k).exists()}
    gb = sum(f.stat().st_size for f in BANCO_DIR.glob('*.npz')) / 1024**3
    print(f'Banco listo: {len(ok_keys)} videos | {gb:.1f} GB en disco | extracción de esta ejecución: {(time.time() - t0) / 3600:.2f} h')
    if filas:
        display(pd.DataFrame(filas).origen.value_counts().rename('videos por origen').to_frame())
else:
    VER_CSV = BASE_DIR / 'verificacion_videos.csv'
    prev = pd.read_csv(VER_CSV) if VER_CSV.exists() else pd.DataFrame(columns=['s3_key', 'ok'])
    prev = prev[prev.s3_key.isin(claves)]
    pendientes = sorted(set(claves) - set(prev.loc[prev.ok.astype(bool), 's3_key'])) if VERIFICAR_VIDEOS else []
    try:
        sizes = s3_sizes(sorted({s3_key_efectiva(k).split('/')[0] + '/' for k in claves}))
        total_gb = sum(sizes.get(s3_key_efectiva(k), 0) for k in claves) / 1024**3
        print(f'Videos a usar: {len(claves)} | {total_gb:.1f} GB en S3 | caché disponible {CACHE_MAX_GB:.1f} GB')
        if total_gb > CACHE_MAX_GB:
            print('No caben todos en el disco: se descargarán de nuevo en cada época. Más rápido: USAR_BANCO_FRAMES=True.')
    except ClientError:
        pass
    if pendientes:
        with ThreadPoolExecutor(HILOS_VERIFICACION) as ex:
            nuevos = list(tqdm(ex.map(verificar_video, pendientes), total=len(pendientes), desc='Verificando videos'))
        ver = pd.concat([prev[~prev.s3_key.isin(pendientes)], pd.DataFrame(nuevos)], ignore_index=True)
        ver.to_csv(VER_CSV, index=False)
    else:
        ver = prev if len(prev) else pd.DataFrame({'s3_key': claves, 'ok': True, 'motivo': ''})
    ver['ok'] = ver.ok.astype(bool)
    ok_keys = set(ver.loc[ver.ok, 's3_key'])

if FALLOS_CSV.exists():
    fallos = pd.read_csv(FALLOS_CSV, header=None, names=['fecha', 'contexto', 's3_key', 'error'])
    reinc = fallos.s3_key.value_counts()
    reinc = set(reinc[reinc >= EXCLUIR_SI_FALLA_N].index)
    if reinc:
        print(f'Excluidos por fallar {EXCLUIR_SI_FALLA_N}+ veces durante entrenamientos anteriores: {len(reinc)}')
        ok_keys -= reinc

malos = splits[~splits.s3_key.isin(ok_keys)].merge(ver[['s3_key', 'motivo']].drop_duplicates('s3_key'), on='s3_key', how='left')
print(f'\nVideos con falla: {len(malos)} de {len(splits)}')
if len(malos):
    display(pd.crosstab(malos.fuente, malos.split))
    display(malos.motivo.fillna('falla durante entrenamiento').str.split(':').str[0].value_counts().rename('motivo').to_frame())
    display(malos[['split', 'fuente', 's3_key', 'motivo']].head(30))
    malos.to_csv(RUNS_DIR / 'videos_corruptos.csv', index=False)

splits_ok = splits[splits.s3_key.isin(ok_keys)]
df_train, df_val, df_test = (splits_ok[splits_ok.split == s].reset_index(drop=True) for s in ('train', 'val', 'test'))
df_muni = df_muni[df_muni.s3_key.isin(ok_keys)].reset_index(drop=True)
SPLIT_HASH = hashlib.sha1(','.join(sorted(df_test.s3_key)).encode()).hexdigest()[:12]
display(splits_ok.groupby(['split', 'y']).size().unstack(fill_value=0).rename(columns={0: 'y=0', 1: 'y=1'}))
print('Hash del test (identifica la partición en los checkpoints):', SPLIT_HASH)

# Verificación visual: frames de un video de validación tal como los recibe el modelo
x0, y0, k0 = VideoDataset(df_val, False)[0]
assert x0.shape == (3, N_FRAMES, CROP, CROP), x0.shape
v0 = denorm(x0)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, t in zip(axes, np.linspace(0, N_FRAMES - 1, 4).astype(int)):
    ax.imshow(v0[t]); ax.set_title(f'segmento {t + 1}'); ax.axis('off')
fig.suptitle(f'{Path(k0).name} | y={int(y0)}'); plt.show()

## 6. Arquitectura X3D-Medium y propuesta de 3 versiones

**X3D (Feichtenhofer, 2020).** Parte de una red 2D mínima y la expande paso a paso, un eje a la vez (frecuencia de muestreo, número de frames, resolución, ancho, ancho del *bottleneck* y profundidad), eligiendo en cada paso el eje que más mejora la precisión por unidad de cómputo. El resultado es una red 3D delgada y eficiente: unos 3 M de parámetros y ~4.9 GFLOPs por clip de $16\times224^2$, contra ~66 GFLOPs del TSN (16 pasadas de ResNet-50).

Cada bloque es un *bottleneck* $1\times1^2\rightarrow3\times3^2$ *depthwise* $\rightarrow1\times1^2$, con *Squeeze-and-Excitation* y activación *swish*:

$$\mathbf{h}_{l+1}=\mathbf{h}_l+\mathcal{F}(\mathbf{h}_l),\qquad \mathcal{F}=\mathrm{Conv}^{1\times1^2}\circ\mathrm{swish}\circ\mathrm{SE}\circ\mathrm{DWConv}^{3\times3^2}\circ\mathrm{Conv}^{1\times1^2}.$$

| Etapa | Bloques | Canales | Salida $T\times H\times W$ |
|---|---|---|---|
| `blocks[0]` stem | — | 24 | $16\times112^2$ |
| `blocks[1]` res2 | 3 | 24 | $16\times56^2$ |
| `blocks[2]` res3 | 5 | 48 | $16\times28^2$ |
| `blocks[3]` res4 | 11 | 96 | $16\times14^2$ |
| `blocks[4]` res5 | 7 | 192 | $16\times7^2$ |
| `blocks[5]` cabeza | conv5 → pool → fc 2048 → **fc 1 (nueva)** | 432 → 2048 → 1 | 1 |

Se parte de los pesos preentrenados en Kinetics-400. La proyección de 400 clases se reemplaza por una neurona (*logit* $z$, $p=\sigma(z)$) y el *pooling* de la cabeza por uno adaptativo global (idéntico para 16 frames). La pérdida es la entropía cruzada binaria, con *label smoothing* opcional igual al del TSN V4:

$$\mathcal{L}=-\frac1n\sum_i\big[\tilde y_i\log\sigma(z_i)+(1-\tilde y_i)\log(1-\sigma(z_i))\big],\qquad \tilde y_i=y_i(1-s)+\tfrac{s}{2}.$$

### Propuesta de 3 versiones
Todas comparten datos, frames, dimensiones, batch efectivo (8), AdamW y la Fase 1 (5 épocas, solo la neurona de salida, $\eta=10^{-3}$). Cada versión prueba **una hipótesis**:

| | V1: Metodología | V2: Ajuste fino completo | V3: Receta del TSN V4 |
|---|---|---|---|
| Hipótesis | protocolo de la metodología 3.2 | el cambio de dominio (Kinetics → CCTV) requiere adaptar también las etapas bajas | la diferencia con el TSN se debe a la arquitectura y no a la receta de entrenamiento |
| Fase 2: etapas entrenadas | res4, res5 + cabeza | **todas** + cabeza | res4, res5 + cabeza |
| Fase 2: épocas y $\eta$ | 15, $10^{-5}$ | 15, $10^{-4}$ | 10, $10^{-5}$ |
| Programa de $\eta$ | coseno | coseno | ReduceLROnPlateau (factor 0.5, paciencia 2) |
| Dropout | 0.5 | 0.5 | 0.4 |
| Label smoothing | — | — | 0.1 |
| Early stopping | — | — | paciencia 3 (val_loss o val_AUC) |

V3 replica los hiperparámetros del TSN V4: si X3D-M lo supera con la misma receta, la ganancia se atribuye a la arquitectura espaciotemporal. En las tres, las estadísticas de BatchNorm quedan congeladas (lotes de 4 videos) y se recorta la norma del gradiente.

La celda construye cada versión, cuenta los parámetros entrenables por fase y verifica una pasada hacia adelante.

In [ ]:
resumen_arq = []
for v, cfg in VERSIONES.items():
    net = build_x3d(cfg['dropout']).to(DEVICE)
    fila = {'versión': v, 'nombre': cfg['nombre'], 'parámetros totales': sum(p.numel() for p in net.parameters())}
    for ph in (1, 2):
        set_phase(net, ph, cfg['etapas_fase2'])
        fila[f'entrenables fase {ph}'] = sum(p.numel() for p in net.parameters() if p.requires_grad)
    net.eval()
    with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.float16, enabled=AMP):
        z0 = net(x0[None].to(DEVICE)).reshape(-1)
    assert z0.shape == (1,) and torch.isfinite(z0).all()
    resumen_arq.append(fila); del net
    if AMP: torch.cuda.empty_cache()
display(pd.DataFrame(resumen_arq).set_index('versión'))
print('Forward OK en las 3 versiones.')

## 7. Entrenamiento del modelo V1
V1: protocolo de la metodología: Fase 1 (5 épocas, solo la neurona de salida) y Fase 2 (15 épocas, res4, res5 y cabeza, $\eta=10^{-5}$ con coseno).

Se puede interrumpir en cualquier momento: al volver a ejecutar la celda, continúa desde la última época completa. El avance queda en `RUNS_DIR/V1/train_log.txt`.

In [ ]:
hist_V1 = train_version('V1')

## 8. Entrenamiento del modelo V2
V2: ajuste fino completo: igual que V1, pero en la Fase 2 se entrenan todas las etapas con $\eta=10^{-4}$.

Se puede interrumpir en cualquier momento: al volver a ejecutar la celda, continúa desde la última época completa. El avance queda en `RUNS_DIR/V2/train_log.txt`.

In [ ]:
hist_V2 = train_version('V2')

## 9. Entrenamiento del modelo V3
V3: receta del TSN V4: Fase 2 de 10 épocas, $\eta=10^{-5}$ con ReduceLROnPlateau, dropout 0.4, label smoothing 0.1 y early stopping con paciencia 3.

Se puede interrumpir en cualquier momento: al volver a ejecutar la celda, continúa desde la última época completa. El avance queda en `RUNS_DIR/V3/train_log.txt`.

In [ ]:
hist_V3 = train_version('V3')

## 10. Métricas de desempeño por época
Para cada versión se grafican, en entrenamiento y validación, la pérdida BCE, AUC-ROC, F1, precisión, *recall* y *accuracy* (umbral 0.5), más el tiempo por época y el pico de VRAM.

- **Línea gris punteada:** paso de la Fase 1 a la Fase 2.
- **Línea amarilla:** mejor época (mayor AUC de validación).
- Una brecha creciente entre entrenamiento y validación, con la pérdida de validación subiendo, indica sobreajuste.
- Las métricas de entrenamiento se calculan durante la misma época (con *dropout* y pesos cambiando), por lo que son algo pesimistas.

El último gráfico superpone el AUC de validación de las tres versiones.

In [ ]:
HIST = {}
for v, cfg in VERSIONES.items():
    f = RUNS_DIR / v / 'history.csv'
    if not f.exists():
        print(f'{v}: sin entrenamiento todavía.'); continue
    HIST[v] = pd.read_csv(f)
    plot_history(HIST[v], f'X3D-M {v}: {cfg["nombre"]}', cfg['ep1'], RUNS_DIR / v / 'indicadores_por_epoca.png')
    cols = ['epoch', 'fase', 'lr', 'train_loss', 'val_loss'] + [f'val_{k}' for k in METRICAS] + ['seg_entrenamiento', 'vram_mb']
    display(HIST[v][cols].round(4).set_index('epoch'))

if HIST:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    for c, (v, h) in zip(PALETA, HIST.items()):
        axes[0].plot(h.epoch, h.val_auc, 'o-', color=c, label=v)
        axes[1].plot(h.epoch, h.val_loss, 'o-', color=c, label=v)
    axes[0].set(title='AUC de validación por época', xlabel='época', ylim=(0, 1.02)); axes[0].legend()
    axes[1].set(title='Pérdida de validación por época', xlabel='época'); axes[1].legend()
    fig.tight_layout(); fig.savefig(RUNS_DIR / 'versiones_por_epoca.png', dpi=150); plt.show()

## 11. Comparación de las 3 versiones y selección del mejor modelo
La selección se hace en **validación**, no en test: el test queda reservado para la comparación final con el TSN (sección 14).

1. Se carga el mejor checkpoint de cada versión y se evalúa en validación, con IC 95 % por bootstrap.
2. Se comparan los AUC por pares con la prueba de DeLong y corrección de Holm.
3. Se resumen los costos de entrenamiento.
4. Se elige la versión de mayor AUC en validación. Si su diferencia con otra no es significativa, la tabla de costos permite justificar en el informe la alternativa más barata.

Al final se evalúa el modelo elegido en test, una sola vez, para las secciones 12 y 14.

In [ ]:
val_loader = make_loader(df_val, False, 'val-comparacion')
PRED_VAL, tablas, costos_ent = {}, {}, []
for v, cfg in VERSIONES.items():
    f = RUNS_DIR / v / 'best.pt'
    if not f.exists():
        print(f'{v}: sin best.pt; se omite.'); continue
    st = torch.load(f, map_location='cpu', weights_only=False)
    net = build_x3d(cfg['dropout']).to(DEVICE); net.load_state_dict(st['model'])
    PRED_VAL[v] = predict(net, val_loader, f'{v} validación').set_index('s3_key')
    del net
    if AMP: torch.cuda.empty_cache()
    h = HIST.get(v, pd.DataFrame(st['hist']))
    costos_ent.append({'versión': v, 'mejor época': st['best_ep'], 'épocas entrenadas': len(h),
                       'min por época': h.seg_entrenamiento.mean() / 60, 'horas totales': h.seg_entrenamiento.sum() / 3600,
                       'VRAM pico (MB)': h.vram_mb.max()})

keys = sorted(set.intersection(*(set(d.index) for d in PRED_VAL.values())))
y_val = next(iter(PRED_VAL.values())).loc[keys].y_true.to_numpy()
tablas = {v: bootstrap_table(y_val, d.loc[keys].p) for v, d in PRED_VAL.items()}
display(pd.DataFrame({f'{v}: {VERSIONES[v]["nombre"]}': fmt_ci(t) for v, t in tablas.items()}).T)
display(pd.DataFrame(costos_ent).set_index('versión').round(2))

pares = []
for a, b in combinations(PRED_VAL, 2):
    a1, a2, z, p = delong_test(y_val, PRED_VAL[a].loc[keys].p, PRED_VAL[b].loc[keys].p)
    pares.append({'A': a, 'B': b, 'AUC A': a1, 'AUC B': a2, 'z': z, 'p DeLong': p})
if pares:
    pares = pd.DataFrame(pares); pares['p (Holm)'] = holm(pares['p DeLong'])
    display(pares.round(4))

MEJOR = max(tablas, key=lambda v: tablas[v].loc['auc', 'valor'])
print(f'\nVersión seleccionada: {MEJOR} ({VERSIONES[MEJOR]["nombre"]}) | AUC val {tablas[MEJOR].loc["auc", "valor"]:.4f}')
if len(pares) and (pares[(pares.A == MEJOR) | (pares.B == MEJOR)]['p (Holm)'] >= 0.05).any():
    print('Su ventaja frente a al menos otra versión no es significativa (p Holm >= 0.05): reportarla como empate técnico.')

fig, ax = plt.subplots(figsize=(7, 5))
for c, (v, d) in zip(PALETA, PRED_VAL.items()):
    fpr, tpr, _ = roc_curve(y_val, d.loc[keys].p)
    ax.plot(fpr, tpr, color=c, lw=2.5 if v == MEJOR else 1.2, label=f'{v} (AUC {roc_auc_score(y_val, d.loc[keys].p):.3f})')
ax.plot([0, 1], [0, 1], ':', color='gray'); ax.set(title='ROC en validación', xlabel='FPR', ylabel='TPR'); ax.legend()
fig.tight_layout(); fig.savefig(RUNS_DIR / 'roc_versiones_val.png', dpi=150); plt.show()

x3d_best = build_x3d(VERSIONES[MEJOR]['dropout']).to(DEVICE)
x3d_best.load_state_dict(torch.load(RUNS_DIR / MEJOR / 'best.pt', map_location='cpu', weights_only=False)['model'])
test_loader = make_loader(df_test, False, 'test')
x3d_test = predict(x3d_best, test_loader, f'X3D-M {MEJOR} test')
x3d_test.to_csv(RUNS_DIR / f'pred_test_x3d_{MEJOR}.csv', index=False)

## 12. Aplicabilidad del modelo: 2 ejemplos de cada tipo
Con el mejor X3D-M en el test se eligen, para cada tipo de resultado, los 2 ejemplos en que el modelo está más seguro:

| Tipo | Significado |
|---|---|
| **TP** | violento clasificado como violento |
| **TN** | no violento clasificado como no violento |
| **FP** | falsa alarma |
| **FN** | violencia no detectada |

Cada ejemplo muestra:
- la **muestra** de 16 frames (8 visibles) que efectivamente vio el modelo, leída del banco de frames;
- el **mapa de calor** Grad-CAM espaciotemporal (Selvaraju et al., 2017) sobre la salida de res5;
- la **evidencia en el tiempo**, ubicada en segundos del video original;
- la **predicción** con su probabilidad;
- un **GIF** del video original junto al mapa de calor del segmento más cercano (solo para estos ejemplos se descarga el video original).

**Cómo leer el mapa de calor.** Siempre se explica la evidencia **a favor de violencia**. Su resolución original es de 7×7 por frame, así que localiza regiones gruesas. Mapas concentrados en logos, marcas de tiempo o fondo, en lugar de personas, indican que el modelo aprendió rasgos de la fuente y no de la acción.

Las figuras y GIF quedan en `RUNS_DIR/galeria/`.

In [ ]:
GALERIA = RUNS_DIR / 'galeria'; GALERIA.mkdir(exist_ok=True)
gal = x3d_test.copy()
gal['tipo'] = [tipo_resultado(y, yh) for y, yh in zip(gal.y_true, gal.y_pred)]
gal['confianza'] = np.where(gal.y_pred == 1, gal.p, 1 - gal.p)
display(gal.tipo.value_counts().rename('videos en test').to_frame())
for tipo in ['TP', 'TN', 'FP', 'FN']:
    sub = gal[gal.tipo == tipo].nlargest(N_EJEMPLOS_POR_TIPO, 'confianza')
    if sub.empty:
        print(f'Sin ejemplos {tipo} en test.'); continue
    for i, (_, r) in enumerate(sub.iterrows(), start=1):
        try:
            show_example(x3d_best, r.s3_key, int(r.y_true), f'{MEJOR}-{tipo}{i}', GALERIA)
        except Exception as exc:
            print(f'No se pudo graficar {r.s3_key}: {exc}')

## 13. Carga del modelo TSN
Se carga `modelamiento_version4/checkpoints/best_model.pt` en la misma clase `BaselineTSN_V4` de su script, con carga estricta de pesos: si falta o sobra un solo tensor, se detiene con error.

El checkpoint se guardó con `pickle.dump` y no con `torch.save`, y por eso `torch.load` falla con el error *"Invalid magic number"*. La función de carga lo detecta y lo lee con `pickle`, llevando todos los tensores a CPU.

Luego se verifica que la partición reproducida coincide con la del TSN: su AUC en la validación reproducida debe ser cercano al `best_val_auc` guardado en el checkpoint. Finalmente se grafican las métricas por época del TSN que vienen en el checkpoint, que llegan hasta la época en que se guardó ese mejor modelo.

In [ ]:
tsn, tsn_info, tsn_hist = load_tsn(TSN_S3_KEY)
print('TSN cargado | metadatos:', tsn_info)

tsn_val = predict(tsn, val_loader, 'TSN validación')
auc_tsn_val = metrics(tsn_val.y_true, tsn_val.p)['auc']
ref = tsn_info.get('best_val_auc')
print(f'AUC del TSN en la validación reproducida: {auc_tsn_val:.4f}' + (f' | guardado en el checkpoint: {float(ref):.4f}' if ref is not None else ''))
if ref is not None and np.isfinite(auc_tsn_val):
    if abs(auc_tsn_val - float(ref)) <= 0.03:
        print('Coinciden: la partición reproducida es la del TSN.')
    elif auc_tsn_val > float(ref) + 0.03:
        warnings.warn('El TSN rinde mejor que en su propia validación: probable fuga (la partición no coincide). Revisar el manifest.')
    else:
        print('Diferencia > 0.03: puede deberse a los videos corruptos excluidos o a diferencias de lectura de frames. Revisar.')

if tsn_hist and tsn_hist.get('train'):
    tr, va = pd.DataFrame(tsn_hist['train']), pd.DataFrame(tsn_hist['val'])
    hh = pd.DataFrame({'epoch': range(1, len(tr) + 1), **{f'train_{k}': tr[k] for k in ['loss'] + METRICAS},
                       **{f'val_{k}': va[k] for k in ['loss'] + METRICAS},
                       'seg_entrenamiento': tr.time_seconds, 'seg_validacion': va.time_seconds, 'vram_mb': tr.gpu_memory_mb})
    plot_history(hh, 'TSN V4 (historial guardado en el checkpoint; VRAM = promedio)', 5, RUNS_DIR / 'tsn_indicadores.png')

## 14. Evaluación TSN vs. mejor X3D-Medium en test
Ambos modelos se evalúan sobre **los mismos videos de test**, con los mismos 16 frames y el mismo preprocesamiento. Como X3D-M usó la misma partición que el TSN, **ninguno de los dos vio el test**: la comparación es limpia.

- **Métricas:** AUC-ROC (métrica de decisión) y F1, precisión, *recall* y *accuracy* con $\tau=0.5$, todas con IC 95 % por bootstrap.
- **Prueba de DeLong:** compara los AUC.
- **Prueba de McNemar:** compara los aciertos con el umbral 0.5.

Una diferencia con $p\ge0.05$ se reporta como no concluyente.

Se agregan las métricas por fuente y el costo de cómputo de ambos modelos en la misma T4.

In [ ]:
tsn_test = predict(tsn, test_loader, 'TSN test')
tsn_test.to_csv(RUNS_DIR / 'pred_test_tsn.csv', index=False)
comunes = sorted(set(tsn_test.s3_key) & set(x3d_test.s3_key))
M = {f'X3D-M {MEJOR}': x3d_test.set_index('s3_key').loc[comunes], 'TSN V4': tsn_test.set_index('s3_key').loc[comunes]}
y_t = M['TSN V4'].y_true.to_numpy()
print(f'Videos de test evaluados por ambos modelos: {len(comunes)}')

TEST_TABLAS = {n: bootstrap_table(y_t, d.p) for n, d in M.items()}
display(pd.DataFrame({n: fmt_ci(t) for n, t in TEST_TABLAS.items()}).T)
(nx, dx), (nt, dt) = M.items()
a1, a2, z, p = delong_test(y_t, dx.p, dt.p)
b, c, pm = mcnemar_exact(y_t, dx.y_pred, dt.y_pred)
print(f'DeLong: AUC {nx} {a1:.4f} vs TSN {a2:.4f} | z = {z:.3f} | p = {p:.4f}' + (' -> significativa' if p < 0.05 else ' -> no concluyente'))
print(f'McNemar (τ=0.5): solo X3D acierta {b} | solo TSN acierta {c} | p = {pm:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for col_, (n, d) in zip(PALETA, M.items()):
    fpr, tpr, _ = roc_curve(y_t, d.p); axes[0].plot(fpr, tpr, color=col_, label=f'{n} (AUC {roc_auc_score(y_t, d.p):.3f})')
axes[0].plot([0, 1], [0, 1], ':', color='gray'); axes[0].set(title='ROC en test', xlabel='FPR', ylabel='TPR'); axes[0].legend()
for ax, (n, d) in zip(axes[1:], M.items()): plot_confusion(ax, y_t, d.y_pred, f'{n} (τ=0.5)')
fig.tight_layout(); fig.savefig(RUNS_DIR / 'tsn_vs_x3d_test.png', dpi=150); plt.show()

fuente = splits.set_index('s3_key').fuente
por_fuente = pd.DataFrame([{'fuente': f, 'modelo': n, 'n': len(g), **metrics(g.y_true, g.p)}
                           for n, d in M.items() for f, g in d.join(fuente).groupby('fuente')])
display(por_fuente.set_index(['fuente', 'modelo'])[['n'] + METRICAS].round(3))

COSTOS = pd.DataFrame({nx: costos_modelo(x3d_best), 'TSN V4': costos_modelo(tsn)}).T
display(COSTOS.round(2)); COSTOS.to_csv(RUNS_DIR / 'costos_tsn_vs_x3d.csv')

## 15. Comparación de los modelos en los videos de la municipalidad
Los videos de Pozo Almonte son la prueba de generalización a cámaras reales. Se evalúan con el mismo protocolo: 16 segmentos sobre todo el video, recorte 224 y $\tau=0.5$.

**Origen de las etiquetas**, en este orden:
1. `MUNICIPAL_MANIFEST_S3_KEY` (CSV con `s3_key` y `etiqueta`).
2. Las filas municipales del manifest, si existen.
3. El nombre de la subcarpeta en S3 (por ejemplo `Fight` / `Normal`).

Si no hay etiquetas, se muestran solo las probabilidades de cada modelo por video y su grado de acuerdo.

**Cuidado con la fuga.** Si algún video municipal está en el manifest del TSN y quedó en su train o val, ambos modelos lo vieron: se excluye y se informa cuántos.

En videos de varios minutos, 16 frames repartidos en toda la duración pueden no capturar un evento breve: un FN municipal puede deberse al muestreo y no al modelo.

In [ ]:
muni = df_muni.copy()
if 'split' in muni:
    vistos = muni.split.isin(['train', 'val'])
    if vistos.any():
        print(f'Se excluyen {int(vistos.sum())} videos municipales que quedaron en train/val del TSN.')
    muni = muni[~vistos]
if muni.empty:
    print('No hay videos municipales disponibles para evaluar.')
else:
    muni_eval = muni.assign(y=muni.y.fillna(0).astype(int))
    muni_loader = make_loader(muni_eval, False, 'municipal')
    PM = {f'X3D-M {MEJOR}': predict(x3d_best, muni_loader, 'X3D municipal'), 'TSN V4': predict(tsn, muni_loader, 'TSN municipal')}
    tabla_m = muni[['s3_key', 'y']].set_index('s3_key')
    for n, d in PM.items():
        tabla_m[f'p {n}'] = d.set_index('s3_key').p
    tabla_m = tabla_m.dropna(subset=[f'p {n}' for n in PM])
    cols_p = [f'p {n}' for n in PM]
    acuerdo = ((tabla_m[cols_p] >= THRESHOLD).nunique(axis=1) == 1).mean()
    print(f'Videos municipales evaluados: {len(tabla_m)} | acuerdo entre modelos (τ=0.5): {acuerdo:.1%}')
    tabla_m.to_csv(RUNS_DIR / 'pred_municipal.csv')
    con_et = tabla_m[tabla_m.y.notna()]
    if con_et.y.nunique() == 2:
        y_m = con_et.y.astype(int).to_numpy()
        display(pd.DataFrame({n: fmt_ci(bootstrap_table(y_m, con_et[f'p {n}'])) for n in PM}).T)
        a1, a2, z, p = delong_test(y_m, con_et[cols_p[0]], con_et[cols_p[1]])
        print(f'DeLong municipal: AUC X3D {a1:.4f} vs TSN {a2:.4f} | p = {p:.4f}' + (' -> significativa' if p < 0.05 else ' -> no concluyente'))
        fig, axes = plt.subplots(1, 3, figsize=(19, 5))
        for col_, n in zip(PALETA, PM):
            fpr, tpr, _ = roc_curve(y_m, con_et[f'p {n}'])
            axes[0].plot(fpr, tpr, color=col_, label=f'{n} (AUC {roc_auc_score(y_m, con_et[f"p {n}"]):.3f})')
        axes[0].plot([0, 1], [0, 1], ':', color='gray'); axes[0].set(title='ROC municipal', xlabel='FPR', ylabel='TPR'); axes[0].legend()
        for ax, n in zip(axes[1:], PM): plot_confusion(ax, y_m, (con_et[f'p {n}'] >= THRESHOLD).astype(int), f'{n} municipal')
        fig.tight_layout(); fig.savefig(RUNS_DIR / 'municipal_comparacion.png', dpi=150); plt.show()
    else:
        print('Sin etiquetas de ambas clases: se muestran solo las probabilidades.')
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(tabla_m[cols_p[1]], tabla_m[cols_p[0]], c=[PALETA[0] if y == 1 else PALETA[1] for y in tabla_m.y.fillna(-1)], s=40)
    ax.axhline(THRESHOLD, color='gray', ls=':'); ax.axvline(THRESHOLD, color='gray', ls=':')
    ax.set(xlabel='p TSN V4', ylabel=f'p X3D-M {MEJOR}', xlim=(0, 1), ylim=(0, 1),
           title='Probabilidad por video municipal (rojo: violento, cian: no violento)')
    fig.tight_layout(); fig.savefig(RUNS_DIR / 'municipal_dispersion.png', dpi=150); plt.show()
    display(tabla_m.sort_values(cols_p[0], ascending=False).round(3).head(20))

In [ ]:
resumen = {'modo': MODO, 'gpu': torch.cuda.get_device_name(0) if AMP else 'cpu', 'split_hash': SPLIT_HASH,
           'n': {'train': len(df_train), 'val': len(df_val), 'test': len(df_test), 'municipal': len(df_muni)},
           'videos_corruptos': int(len(malos)), 'versiones': VERSIONES, 'seleccionada': MEJOR,
           'validacion': {v: t.to_dict(orient='index') for v, t in tablas.items()},
           'test': {n: t.to_dict(orient='index') for n, t in TEST_TABLAS.items()},
           'costos': COSTOS.to_dict(orient='index'), 'tsn_auc_val_reproducido': auc_tsn_val, 'tsn_info': tsn_info}
(RUNS_DIR / 'resumen.json').write_text(json.dumps(resumen, ensure_ascii=False, indent=2, default=float))
print('Resumen en', RUNS_DIR / 'resumen.json')
if RESPALDO_S3:
    for f in RUNS_DIR.rglob('*'):
        if f.is_file() and f.name != 'last.pt':
            s3().upload_file(str(f), S3_BUCKET, f"{S3_RESULT_PREFIX}{MODO.lower()}/{f.relative_to(RUNS_DIR).as_posix()}")
    print(f'Respaldo en s3://{S3_BUCKET}/{S3_RESULT_PREFIX}{MODO.lower()}/')

### Referencias
- DeLong, E. R., DeLong, D. M., & Clarke-Pearson, D. L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves: A nonparametric approach. *Biometrics, 44*(3), 837–845. https://doi.org/10.2307/2531595
- Dietterich, T. G. (1998). Approximate statistical tests for comparing supervised classification learning algorithms. *Neural Computation, 10*(7), 1895–1923. https://doi.org/10.1162/089976698300017197
- Feichtenhofer, C. (2020). X3D: Expanding architectures for efficient video recognition. *CVPR*, 203–213. https://arxiv.org/abs/2004.04730
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*, 770–778. https://arxiv.org/abs/1512.03385
- Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*. https://arxiv.org/abs/1711.05101
- Selvaraju, R. R., et al. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. *ICCV*, 618–626. https://arxiv.org/abs/1610.02391
- Sun, X., & Xu, W. (2014). Fast implementation of DeLong's algorithm for comparing the areas under correlated receiver operating characteristic curves. *IEEE Signal Processing Letters, 21*(11), 1389–1393. https://doi.org/10.1109/LSP.2014.2337313
- Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J., & Wojna, Z. (2016). Rethinking the Inception architecture for computer vision (label smoothing). *CVPR*. https://arxiv.org/abs/1512.00567
- Wang, L., et al. (2016). Temporal segment networks: Towards good practices for deep action recognition. *ECCV*, 20–36. https://arxiv.org/abs/1608.00859